# **Author:** Seth M. Woodbury (woodbuse@uw.edu)

# **Organization & Notebook Setup/Initialization (Run Everytime)** 

In [ ]:
######################################################################
### IMPORT PACKAGES & SETUP NOTEBOOK (RUN AT BEGINNING EVERY TIME) ###
######################################################################

project_name     = 'RFdiffusion3_tutorial'

### MANDATORY USER SPECIFICATIONS ###
### REPOSITORY ROOT (auto-detected -- nothing here needs editing) ###
# Walks up from the current directory looking for the repository markers, so a
# fresh clone works anywhere on disk. Override with the ZINC_HYDRO_REPO
# environment variable if you have an unusual layout.
def _find_repo_root():
    import os as _os
    from pathlib import Path as _Path
    _markers = ('Scripts', 'Software', 'Environment', 'LICENSE')
    _start = _Path(_os.environ.get('ZINC_HYDRO_REPO') or _Path.cwd()).resolve()
    for _cand in (_start, *_start.parents):
        if all((_cand / _m).exists() for _m in _markers):
            return str(_cand)
    raise RuntimeError(
        f"Could not find the repository root above {_start}. "
        f"Start Jupyter from inside the cloned repository, or set ZINC_HYDRO_REPO."
    )

github_repo_dir = _find_repo_root()
obabel_path      = "obabel"   # resolved from $PATH inside the conda env; override with $ZINC_HYDRO_OBABEL

### SUBFOLDERS/VARIABLES (OPTIONAL: ADD EXTRA TO LIST OF DEFAULTS) ###
# 1.) subfolders to build in notebook directory
wrk_dirs_list = [
    'theozymes', 'cmds', 'slurm_submit', 'logs', 'rfd3_json', 'graphs' # optionally add more input subdirectories
]

# 2.) subfolders to build in output directory
out_dirs_list = [
    'rfdiffusion3_out', 'predesign_out', # optionally add more output subdirectories
]

###################################################### AUTO SETUP ######################################################
### STANDARD LIBRARY IMPORTS ###
import shlex, glob, json, math, os, random, copy, re, shutil, statistics, string, subprocess, sys
import textwrap, warnings, concurrent.futures, time, multiprocessing, itertools, operator
from itertools import product
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

### DERIVED PATHS (CHANGE AT USER DISCRETION) ###
scripts_dir   = f'{github_repo_dir}/Scripts/'
special_scripts_dir = f'{github_repo_dir}/Scripts/'  # scaffold_handling/ scripts live here
software_dir  = f'{github_repo_dir}/Software/'
working_dir   = f'{github_repo_dir}/RFdiffusion3_Tutorial/'
output_dir    = f'{working_dir}outputs/'

for p in [working_dir, output_dir, scripts_dir, software_dir]: 
    Path(p).mkdir(parents=True, exist_ok=True) # ensure base dirs exist so downstream cells never fail on missing paths

### 3RD PARTY IMPORTS ###
import numpy as np
import pandas as pd

### CUSTOM IMPORTS ###
if scripts_dir not in sys.path:
    sys.path.append(scripts_dir)
import General_NoteBook_Functions as functions
import env_config  # resolves interpreter/container/obabel paths portably

### SETUP SUBDIRECTORIES & CREATE VARIABLES ###
functions.setup_directories(working_dir, wrk_dirs_list, export_globals=True, globals_dict=globals())
functions.setup_directories(output_dir,  out_dirs_list,  export_globals=True, globals_dict=globals())

### DERIVED FROM THE EXPORTED DIRS (setup_directories must run first) ###
params_files_dir = f'{theozymes_dir}params/'   # Rosetta .params for the ligands, if you have them

### OPTIONALS ###
functions.set_pandas_display(all_on=True) # pandas display comfort defaults (toggle here if desired)
os.chdir(working_dir) # move into working dir for relative IO

### PRINTS ###
print(f"### PROJECT {project_name} NOTEBOOK SUCCESSFULLY INITIALIZED ON {datetime.now().strftime('%Y-%m-%d AT TIME %H:%M:%S')} ### ")
print(f"\nExample Variables & Paths:")
print(f"   working_dir   = {working_dir}")
print(f"   theozymes_dir = {theozymes_dir}")
print(f"   output_dir    = {output_dir}")
if not Path(obabel_path).exists():
    print(f"⚠️  obabel not found at: {obabel_path} (you can proceed; update later)")

# **I.I (OPTION 1): Create Theozymes from Existing PDB Structure** 

### I.I.A Find PDB of Interest & Crop the Active Site Residues/Cofactors/Ligands of Interest

For this worked example, we will design a de novo metalloprotease by starting from a pre-existing PDB structure and extracting its active site as input to RFdiffusion3. The source can be (1) a structure-prediction model (e.g., AlphaFold2/AlphaFold3 or Chai-1) or (2) an experimental structure from the Protein Data Bank; ideally, the PDB contains the relevant cofactor(s), ligand(s), and catalytic machinery arranged in a catalytically meaningful geometry. This is non-trivial, but recall that enzymes preferentially stabilize the transition state rather than the substrate(s) or product(s). Consequently, the best starting point is typically a PDB structure with a bound transition-state analog (i.e., a mimic of the transition state), which most closely captures the catalytic residue geometry required for catalysis.

Phosphoesters are classic transition-state analogs for ester- and amide-cleaving metallohydrolases: their tetrahedral geometry and localized negative charge closely mimic the anionic tetrahedral state formed during hydroxide attack on the scissile bond. In the specific case of a zinc protease, peptide substrates bearing a phosphonamidate at the cleavage site are especially effective transition state analogs. A careful search of the Protein Data Bank reveals a crystal structure of the metalloprotease astacin bound to a phosphonamidate transition-state analog (PDB: 1QJI; 2.14 Å resolution, deposited in 1999), which provides an excellent starting point for protein design. You can directly download the PDB from the Protein Data Bank, but we have already done so and saved it with the basename "pdb_00001qji__AstacinZnProtease_w_TSA.pdb".

Now that the PDB is downloaded, we must save a "cropped" version of it which only contains the features necessary for catalysis (i.e., ligands, cofactors, and catalytic residues), which will ultimately be our "theoretical enzyme" (theozyme). This is also non-trivial, especially when considering what residues you want to keep or if you wish to include anything like important water molecules. You should spend a great deal of time considering what choices to make (read the literature too!), as they will stay fixed throughout RFdiffusion3, and we also keep them fixed throughout our design pipeline. We suggest that you make several hypotheses about different combinations of residues that could be important to fix, usually with a lot of overlap.

In this case, we know that H92, H96, and H102 are absolutely crucial to chelate the Zn(II) ion, and we know that E93 is the general base for this hydrolysis reaction. Reading the original publication (https://www.nature.com/articles/nsb0896-671), we also learn that Y149 is important for stabilizing the oxyanion and that M147 is part of a conserved motif that sits below the active site, which may or may not be useful to consider for design. There are also other residues binding the substrate that we could consider, but for simplicity let's keep our focus on H92/E93/H96/H102/M147/Y149, in addition to the Zn(II) and TSA ligands (let's also assume there are no other waters we want to keep). We will take this maximal set of residues we're interested in and crop it, then we can crop this set into smaller subsets as parallel inputs for RFdiffusion3 later, if we wish to do that. Importantly, we will also add necessary hydrogens to our ligands (important) and catalytic residues (less important, really only important for HIS to determine HIS-D or HIS-E) for downstream steps in Rosetta beyond this tutorial, although note that RFdiffusion3 does not require hydrogens during inference.

To perform the cropping, open the PDB in pymol (using version 3 here), select the residues/cofactors/ligands of interest, click sele -> action -> copy to object -> new and then you should see a new object appear which is your theozyme!! You can also select your ligand at this point and click sele -> action -> hydrogens -> add to fill in missing hydrogens, and the same can be done for your catalytic residues, although since the histidines already have hydrogens on their nitrogen atoms determining their zinc coordination (i.e., HD1), we will leave them as is. Then, with only obj01 open click file in the top right corner -> Export Structure -> Export Molecule -> Save -> then change file type to PDB and give it a name. Congrats, you just saved your first theozyme!! For reference, we called it "pdb_00001qji__theozyme_HEHHYM.pdb".

Now, we will run it through a quick script to clean up its format. Importantly, this script will add information which will be useful for identifying where your catalytic residues went in the sequence after RFdiffusion3 (which is Rosetta-readable) so that you can fix them in sequence or reference them later in your pipeline for manipulations (e.g., fixing their geometry while doing Rosetta relax). 

### I.I.B Format the Theozymes

STEP 1: "Clean" the PDB files such that we will make REMARK 666 lines, remove protein hydrogens, and combine all separate ligand(s)/cofactor(s) into one ligand complex with a name of your choice. The following cell will generate the command for you to run, then you must copy and execute this command in your terminal.

In [ ]:
##############################################
### CLEAN PDB → THEOZYME (COMMAND BUILDER) ###
##############################################

### INPUTS ###
input_pdb   = f"{theozymes_dir}from_PDB_structure/pdb_00001qji__theozyme_HEHHYM.pdb"
output_pdb  = f"{theozymes_dir}from_PDB_structure/step1__cleaning/pdb_00001qji__theozyme_HEHHYM__lig_TSA.pdb"

### OPTIONAL ORDERING FOR CATALYTIC RESIDUE REMARK 666 LINES | FORMAT: "ChainNumber" (e.g., A11 for residue 11 on chain A) ###
remark_front = ["A92", "A93", "A96", "A102"]  # e.g., ["A244","A199"]
remark_back  = ["A147", "A149"]  # e.g., ["A207","A143"]

### LIGAND COMBINATION TOGGLE ###
combine_ligands = True  # True → unify all HETATMs into a single ligand named by ligand_code
ligand_code     = "TSA" if combine_ligands else None  # set to None when not combining

### VALIDATION ###
if combine_ligands and not ligand_code:
    raise ValueError("When combining ligands, you must set `ligand_code` (3-letter).")
if (not combine_ligands) and (ligand_code is not None):
    raise ValueError("`--no_combine_ligands` is set; `ligand_code` must be None.")

### CONSTANTS ###
combined_script = f"{scripts_dir}prepare_PDB_structure_into_theozyme.py"

### BUILD COMMAND ###
cmd = ["python", combined_script, "--input_pdb", input_pdb, "--output_pdb_path", output_pdb,]

if combine_ligands:
    cmd += ["--ligand_complex_3_letter_name", ligand_code]
else:
    cmd.append("--no_combine_ligands")
if remark_front:
    cmd += ["--remark666_residue_front_order"] + remark_front
if remark_back:
    cmd += ["--remark666_residue_back_order"] + remark_back

### CREATE OUTPUT DIR ###
os.makedirs(os.path.dirname(output_pdb) or ".", exist_ok=True)

### PRINT COMMAND ###
command_str = " ".join(shlex.quote(tok) for tok in cmd)
print(command_str)

### I.I.C (OPTIONAL) Split the Theozymes into Subsets of Residues

STEP 2 (OPTIONAL): Split the theozyme into different subsets of residues for different inputs to RFdiffusion3. This can also be done at the argument step for RFdiffusion3 (i.e., do not fix certain residues when formulating that command), but we often find it more convenient to split the theozymes into separate inputs if we wish to try multiple approaches with RFdiffusion3. As inputs for our specific case, we will consider the essential catalytic machinery (HEHH), the essential catalytic machinery + oxyanion hole (HEHHY), and the essential catalytic machinery + oxyanion hole + hypothesized important Met residue (full theozyme; HEHHYM). Run the cell below to generate your commands for splitting the theozyme and execute them in your terminal.

In [ ]:
###############################################################
### PDB RESIDUE FILTER → COMMAND BUILDER (KEEP or REMOVE)   ###
###############################################################

### INPUTS ###
input_pdb_to_split = f"{theozymes_dir}from_PDB_structure/step1__cleaning/pdb_00001qji__theozyme_HEHHYM__lig_TSA.pdb"
output_pdb         = f"{theozymes_dir}from_PDB_structure/step2__splitting/pdb_00001qji__theozyme_HEHH__lig_TSA.pdb"

### MODE (choose exactly one list; leave the other empty) | Use tokens like "A92", "A93" ###
residue_list_to_keep   = ["A92", "A93", "A96", "A102"] #, "A149"] #, "A147"]  # ← keep ONLY these
residue_list_to_remove = []                                            # ← or remove these

### MOST-IMPORTANT TOGGLES ###
auto_keep_ligands                = True   # default ON: auto-keep HETATMs (non-water)
auto_keep_ligand_waters          = False  # include HOH/WAT/DOD in auto-keep
prune_orphan_TER                 = True   # prune TER blocks that would be empty
verbose                          = True   # print detailed summary
dry_run                          = False  # build & show command, skip write when True

### SCRIPT PATH ###
filter_script = f"{scripts_dir}split_theozyme_into_subsets.py"  # your filter script

### VALIDATION ###
if bool(residue_list_to_keep) == bool(residue_list_to_remove):
    raise ValueError("Specify exactly one: non-empty residue_list_to_keep OR non-empty residue_list_to_remove.")

### BUILD COMMAND ###
cmd = ["python", filter_script, "--input_pdb_to_split", input_pdb_to_split,"--output_pdb", output_pdb]

if residue_list_to_keep:
    cmd += ["--residue_list_to_keep"] + residue_list_to_keep
else:
    cmd += ["--residue_list_to_remove"] + residue_list_to_remove

# Map toggles to CLI flags
if not auto_keep_ligands:
    cmd.append("--do_not_automatically_keep_ligands__SPECIFY_THEM_IN_LIST")
if auto_keep_ligand_waters:
    cmd.append("--auto_keep_hetatm_include_water")
if not prune_orphan_TER:
    cmd.append("--no_prune_orphan_TER")
if verbose:
    cmd.append("--verbose")
if dry_run:
    cmd.append("--dry_run")

### CREATE OUTPUT DIR ###
os.makedirs(os.path.dirname(output_pdb) or ".", exist_ok=True)

### PRINT COMMAND ###
command_str = " ".join(shlex.quote(tok) for tok in cmd)
print(command_str)

### Move to Section II.

# **I.II (OPTION 2): Create Theozymes from Quantum Chemistry Calculation** 

**It is recommended that new users skip this step and go with I.I. for the first run through this tutorial. This is only for preparing quantum chemistry outputs into PDB inputs for RFdiffusion3.**

### I.II.A Perform Quantum Chemistry Calculation

Quantum chemistry is a computational framework that uses the principles of quantum mechanics to predict the energies, forces, geometries, and electronic structures of molecules and molecular systems. Instead of relying on empirical force fields, quantum chemistry explicitly models how electrons are distributed and how atoms interact, allowing accurate evaluation of reaction intermediates, transition states, and catalytic effects. There are many models of varying computational cost and accuracy for calculating these energies, ranging from simple semi-empirical methods to high-level correlated wavefunction approaches. Density functional theory (DFT) occupies the middle of this spectrum and is widely used because it provides a strong balance of accuracy and computational efficiency, especially for systems containing metals like Zn(II) or multi-center charge interactions. However, DFT is not a single method: there are many flavors (“functionals”), each with different approximations and strengths. For example, B3LYP is a classic hybrid functional, M06-2X incorporates empirical corrections tuned for main-group thermochemistry, ωB97X-D includes long-range correction and dispersion, and PBE0 is commonly used for transition metals. Choosing the right functional and basis set is part of ensuring that the theozyme TS geometry you compute is both chemically realistic and computationally tractable.

In theozyme-based enzyme design, quantum chemistry is essential for building a high-fidelity model of the transition state (TS) and its immediate chemical environment (primary sphere), since enzymes should preferentially stabilize the transition state over the substrate and product. In this worked example, we will use one of the quantum chemistry calculations that was performed for design campaign #2, recreating the theozyme which ultimately became the input used to generate ZETA_2 with RFdiffusion3! Here, the choice of softwares used is up to the user - we used the ChemCraft (a commercial software at https://www.chemcraftprog.com/) as our chemical modeling software to construct models of initial guesses of the TS, and we used Gaussian (a commercial software at https://gaussian.com/) to perform our quantum chemistry calculations and TS optimization. Because these are not open-source, we do not have any submodules contained within this GitHub repo for quantum chemistry, and instead only show the inputs/outputs from this process.

We begin by assembling a small molecular model (“theozyme”) of our hypothesized active site stabilizing the transition state of the rate-limiting step of the hydrolysis reaction on our ester substrate (substrate = 4MU-PA | rate limiting step = nucleophilic attack of the ester). This theozyme contains the substrate, the Zn(II), and the reacting side-chain functional groups acting as the general base and Zn(II)-ligands (i.e., His:Imidazole-CB, Glu: AcO⁻). This cluster is constructed manually in ChemCraft, where we place and orient atoms to approximate the expected TS geometry (bond distances, angles, Zn coordination, nucleophile alignment, etc.). Because Gaussian will optimize the structure toward the closest stationary point on the potential-energy surface, our initial guess must already resemble the true TS; if it is too far away, the optimizer will collapse to the reactant, the product, or a non-physical geometry. Thus, this often takes great chemical intuition, mechanistic knowledge, and sometimes inspiration from previous, similar chemical reactions in the literature. For us, this initial guess can be found in the `Gaussian_TSopt` subdirectory as a .xyz file called `ZETA_2__DFTqc_theozyme_initial_guess_for_Gaussian.xyz`.

Once the initial geometry is assembled in ChemCraft (and we saved a .xyz as a reference), we generate the Gaussian input (.com) file, specifying the procedure, level of theory, charge, multiplicity, solvent, and any atoms we intend to freeze. In our example (.com file called `ZETA_2__DFTqc_theozyme_TSopt_Gaussian.com`), this involves defining a ground-state optimization, holding the forming bond fixed (i.e., hydroxide-O attacking ester-C), followed by a linked transition-state search, supplying the mixed basis set (6-31G(d) for main-group atoms and SDD for Zn), including dispersion corrections, and using CPCM(water) solvation. The ModRedundant section encodes any geometric constraints (e.g., frozen distances or angles), and the second Link1 step reads in the optimized geometry to perform the TS optimization and frequency check. In simple terms, this procedure finds the energetic saddle point around our initial guess (lowest energy TS). This fully defines the quantum-chemical workflow used to refine the theozyme into a validated TS structure suitable for downstream enzyme design.

Gaussian then performs energy minimization or TS optimization, computing the forces and (when needed) the Hessian to iteratively adjust the geometry. The result is a fully optimized quantum-mechanical TS model whose key distances and angles define the catalytic constraints you then translate into the RFdiffusion3 input of your enzyme-design pipeline. Note that Gaussian does not always converge to the TS, and sometimes this will take multiple attempts from different initial guess conformation geometries. In our output, which is a .log file called `ZETA_2__DFTqc_theozyme_TSopt_Gaussian.log`, you can view the path that Gaussian took for 1.) optimizing the ground-state geometry with the forming bond-length fixed, and 2.) optimizing the TS geometry by unlocking the forming bond, and 3.) computing the vibrational frequencies, where the presence of a single negative (imaginary) frequency confirms that the optimized structure is a first-order saddle point (a true TS).

Below is a cell which constructs a simple slurm submission (.sh) file for executing a Gaussian job, if you have Gaussian and the slurm setup. These are the parameters we used to run this calculation.

In [ ]:
###########################################################
### AUTOMODIFY .com IN-PLACE & AUTO-GENERATE .sh SCRIPT ###
###########################################################

### INPUTS ###
com_file         = f"{theozymes_dir}from_quantum_chemistry/Gaussian_TSopt/ZETA_2__DFTqc_theozyme_TSopt_Gaussian.com"
email_for_output = "EXAMPLE_EMAIL@uw.edu"  # Input your email for SLURM output notifications
mem_per_cpu      = "4G"                    # Memory per CPU for SBATCH (4 GB × 4 CPUs = 16 GB total)
nproc            = 4                       # Number of CPU cores for Gaussian/OpenMP
time_limit       = "24:00:00"              # Slurm wall-time limit

### CONSTANTS ###
gaussian_path   = os.environ.get("GAUSS_EXEDIR", "/path/to/gaussian/g16/")  # your Gaussian install; Gaussian is licensed commercial software

### DERIVED PATHS ###
com_path = Path(com_file)
base     = com_path.stem
com_dir  = com_path.parent

### SLURM SCRIPT TEMPLATE ###
SLURM_TEMPLATE = f"""#!/bin/bash
#SBATCH --job-name={base}
#SBATCH --nodes=1
#SBATCH --ntasks={nproc}
#SBATCH --cpus-per-task=1
#SBATCH --mem-per-cpu={mem_per_cpu}
#SBATCH --time={time_limit}
#SBATCH --output={logs_dir}slurm_%j.out
#SBATCH --error={logs_dir}slurm_%j.err
#SBATCH --mail-type=END
#SBATCH --mail-user={email_for_output}
#SBATCH --chdir={com_dir}

### EXPORT GAUSSIAN ###
export GAUSS_EXEDIR='{gaussian_path}'

### EXPORT TEMPORARY SCRATCH DIRECTORY ###
SCRATCH="${{SLURM_TMPDIR:-${{TMPDIR:-/tmp}}}}/gauss_scratch"
mkdir -p "$SCRATCH" || {{ echo "Failed to make $SCRATCH"; exit 1; }}
export GAUSS_SCRDIR="$SCRATCH"

### GATHER INFO ###
echo "Running on node: $HOSTNAME"
echo "Scratch dir: $GAUSS_SCRDIR"
echo "This is task $SLURM_ARRAY_TASK_ID, which will do runs $START_NUM to $END_NUM"
lscpu | grep "Model name"
free -h

### EXECUTE GAUSSIAN TASK ###
filename="{base}"
"$GAUSS_EXEDIR/g16" < "$filename.com" > "$filename.log"
"""

### MODIFY .com IN-PLACE ###
# Compute total Mem for Gaussian input (in GB)
total_mem_gb = int(mem_per_cpu.rstrip("G")) * nproc

modified_lines = []
for line in com_path.read_text().splitlines():
    if line.startswith("%chk="):
        modified_lines.append(f"%chk={base}")
    elif line.startswith("%Mem="):
        modified_lines.append(f"%Mem={total_mem_gb*1000}MB")
    elif line.startswith("%NProc="):
        modified_lines.append(f"%NProcShared={nproc}")
    elif line.startswith("%NProcShared="):
        modified_lines.append(f"%NProcShared={nproc}")
    else:
        modified_lines.append(line)

# Write updated .com
com_path.write_text("\n".join(modified_lines) + "\n")

### WRITE SLURM SCRIPT ###
Path(slurm_submit_dir).mkdir(parents=True, exist_ok=True)
sh_path = Path(slurm_submit_dir) / f"{base}.sh"
sh_path.write_text(SLURM_TEMPLATE)

### SUMMARY OUTPUT ###
print(f"### Modified Gaussian Input ###")
print(f"{com_path}")
print(f"\n### SLURM Script ###")
print(f"sbatch {sh_path}")

### I.II.B Grab Optimized Transition State Geometry

Now that our DFT quantum-chemistry optimization of the theozyme transition-state geometry has converged, we can convert the optimized structure into a PDB input suitable for RFdiffusion3. This step is necessarily a bit manual: I prefer to group the non-protein atoms (substrate, Zn(II), waters/solvent, and any atypical hydrogens) under a single 3-letter ligand code, while the protein-derived components of the theozyme (the side-chain analogs) must be assigned their canonical PDB atom names (https://www.cgl.ucsf.edu/chimera/docs/UsersGuide/tutorials/pdbintro.html), and each residue assigned an arbitrary but unique combination of residue number and chain identifier (e.g., A1, A2, A3 or A1, B2, C3 or A1, B1, C1). Note that you can have 2+ ligands as inputs to RFdiffusion3, but I find it easier to work with Rosetta downstream if the non-protein atoms are just one "ligand complex" so they move together as a unit, but this is entirely up to the user.

To streamline this, I first export the optimized geometry as an .xyz file and reorder it so that all ligand atoms appear as one continuous block (e.g., atoms 1–10). I then convert this .xyz file to a preliminary PDB using a tool such as Open Babel, and afterwards make the necessary manual edits: renaming residues, adjusting atom labels, assigning chains, and ensuring formatting correctness, using a text editor like Sublime Text. I leave these steps up to the user to complete with whatever in-silico tools or methods they prefer. However, **note** that each side chain must include a CA atom, even if you do not intend to fix it; this serves only as a placeholder for RFdiffusion3, and its coordinate can be arbitrary.

Below is a script designed to process Gaussian .log files and extract optimized transition-state geometries. It scans each file, verifies that the optimization converged, and checks that exactly one imaginary frequency is present, indicating a valid first-order saddle point, before generating a corresponding .xyz file. You can always retrieve these coordinates manually (e.g., by opening the .log file in GaussView or ChemCraft and locating the final geometry in the frequency section), but this script automates that workflow for large batches of calculations. This tool is especially useful when working with directories containing many Gaussian runs of the same molecular system (e.g., different local conformations or sampling replicas) where the atom ordering and identities remain consistent. In such cases, you can optionally specify atom-index ranges corresponding to the non-protein atoms (such as metals, ligands, or solvent), allowing the script to reorder them to the front of the .xyz output for easier downstream theozyme handling. As always, be mindful that this parsing logic reflects standard Gaussian output conventions, but details may vary depending on the specific calculation setup. **Note** that you must have obabel setup to run this (https://github.com/openbabel/openbabel).

In [ ]:
#############################################################################
### GENERATE PYTHON COMMAND FOR PARSING A SINGLE GAUSSIAN LOG FILE TO XYZ ###
#############################################################################

### INPUTS ###
gaussian_log_file = f"{theozymes_dir}from_quantum_chemistry/Gaussian_TSopt/ZETA_2__DFTqc_theozyme_TSopt_Gaussian.log"

### OPTIONAL INPUTS ###
reorder_ligand_atoms_flag = "1-18,43-43,51-71"  # Specify atom ranges, leave empty/None for no reordering
ignore_TS_warning_flag    = False               # Whether to autopass if frequency analysis is sketchy (--ignore_TS_warning)

output_path_override      = f"{theozymes_dir}from_quantum_chemistry/ZETA_2__theozyme_optTS_geom.xyz" # e.g., "/absolute/path/to/my_TS_output.xyz" | If output_path_override is set, it will be passed as --output_path and will TAKE PRECEDENCE over output_dir_override and output_basename_override.
output_dir_override       = None                                                 # e.g., f"{theozymes_dir}from_quantum_chemistry/Gaussian_TSopt/xyz_outputs"
output_basename_override  = None                                                 # e.g., "my_custom_basename"

### OBABEL PATH | NOTE: YOU MUST INSTALL OPENBABEL ON YOUR OWN ###
obabel_exe_path           = "obabel" # Uses obabel from active conda env; or full path (e.g., /opt/conda/envs/zinc_hydro/bin/obabel)

### CONSTANTS ###
script_path = f"{scripts_dir}parse_gauss_log_files_make_xyz.py"

### COMMAND GENERATION ###
commands = []
output_path_warning_emitted = False

if gaussian_log_file:
    log_path = gaussian_log_file
    command = f"python {script_path} --input_gauss_log {log_path}"

    # Add optional flags if specified
    if reorder_ligand_atoms_flag:
        command += f" --reorder_ligand_atoms_first {reorder_ligand_atoms_flag}"
    if obabel_exe_path:
        command += f" --obabel_exe {obabel_exe_path}"
    if ignore_TS_warning_flag:
        command += " --ignore_TS_warning"
    if output_path_override:
        # Precedence + warning
        if (output_dir_override or output_basename_override) and not output_path_warning_emitted:
            print("### WARNING: output_path_override is set. --output_dir and --output_basename overrides will be IGNORED. ###")
            output_path_warning_emitted = True
        command += f" --output_path {output_path_override}"
    else:
        # Only use these if output_path_override is NOT set
        if output_dir_override:
            command += f" --output_dir {output_dir_override}"
        if output_basename_override:
            command += f" --output_basename {output_basename_override}"
    commands.append(command)

### PRINT COMMANDS ###
print("### COMMANDS TO RUN ###")
for command in commands:
    print(f"{command}\n")

After executing the above command in your terminal, you should have a .xyz file where all of the ligand atoms are grouped together. This file should be called `ZETA_2__theozyme_optTS_geom.xyz` in our example. Again, you could also just grab these coordinates manually from the .log file in a chemical modeling software.

### I.II.C Convert Optimized Transition State Geometry .xyz File into .pdb File

As mentioned in the previous section, the next (and least trivial) task is converting our theozyme into a `.pdb` file suitable for RFdiffusion3. We also need to insert a “placeholder” CA atom, even if we do not plan to fix it—so that RFdiffusion3 can decide its final placement. This could be done manually, but I’ve written a small driver cell that automatically assembles a single command to run my end-to-end `theozyme_XYZ_to_PDB__MAIN.py` pipeline.

This command triggers a multi-step workflow: splitting the theozyme into residue and ligand fragments, converting them with Open Babel, identifying which amino acid each fragment corresponds to, pulling the idealized Rosetta residue for each fragment, superimposing that ideal Rosetta residue onto the DFT-derived subset of atoms, installing any missing atoms to produce a complete canonical residue, assigning proper PDB atom names, and standardizing GLU/ASP OE/OD labels based on proximity to a specified ligand atom. In simple terms, this workflow keeps all quantum-chemistry-optimized atoms exactly fixed while filling in the missing atoms only as placeholders, ensuring that every residue receives correct chain labels, residue numbers, and canonical atom naming.

The cell below collects all required inputs (XYZ file, ligand code, ligand atom ranges, and the tip-residue identities), along with optional debugging/cleanup flags and tool-path overrides, and then prints a fully assembled command that you can copy, paste, or run directly. A few cautions: this conversion pipeline is still in beta, relies on Open Babel and an Apptainer container being properly installed, and may require occasional debugging (e.g., checking intermediate PDBs, preserving temporary files, or adjusting residue-matching thresholds). For first-time runs, I strongly recommend enabling the “keep temp files” options so that you can inspect the intermediate fragment PDBs and troubleshoot if something looks off. 

In [ ]:
############################################################################################
### GENERATE SINGLE PYTHON COMMAND FOR CONVERTING .xyz TO .pdb WITH SPECIFIED PARAMETERS ###
############################################################################################

### FROM PREVIOUS CELL ###
optimized_xyz_dir = f"{theozymes_dir}from_quantum_chemistry/"  # Path to theozyme XYZs
ligand_atom_range = "1-40"                                     # Atom range for ligand atoms (in the XYZ)

### REQUIRED INPUTS ###
input_xyz_file            = f"{optimized_xyz_dir}ZETA_2__theozyme_optTS_geom.xyz"
ligand_3letter_code       = "SZD"                              # 3-letter code for the ligand
ligand_chain              = "Z"                                # Chain ID for ligand (matches script default)
tip_atom_residues_3letter = ["his", "his", "his", "glu"]       # 3-letter residue codes for theozyme tip residues

### OPTIONAL BUT RECOMMENDED ###
ligand_atom_for_proximity_flag = "H1"  # e.g., "H1" or None | Specify the ligand atom for proximity evaluation (GLU/ASP OE1/OE2 or OD1/OD2 standardization), or set to None to skip this step.

### OPTIONAL FLAGS (BOOLEANS) ###
DO_NOT_pass_tip_atom_residues_flag  = False    # If True: do NOT pass residue tips to identifier script
keep_temp_smiles_flag               = False    # If True: keep SMILES and residue PDB temp files
keep_temp_residue_ligand_files_flag = False    # If True: keep intermediate residue & ligand TEMP PDBs

### OPTIONAL TOOL PATH OVERRIDES (ONLY IF YOUR MAIN SCRIPT SUPPORTS THESE ARGS) ###
apptainer_path_override = None   # None -> run with the active python (conda activate zinc_hydro).
# Set to a container path, or export ZINC_HYDRO_SIF, to run inside Apptainer instead.
# Build one with: apptainer build zinc_hydro.sif Environment/zinc_hydro.def
obabel_path_override    = "obabel"                  # Uses obabel from active conda env; or full path (e.g., /opt/conda/envs/zinc_hydro/bin/obabel)

### CONSTANTS ###
script_path = f"{software_dir}theozyme_XYZ_2_PDB__beta/theozyme_XYZ_to_PDB__MAIN.py"

################################
### BUILD COMMAND AS A LIST  ###
################################

cmd_parts = ["python", script_path, "--input_xyz", input_xyz_file, "--ligand_atom_ranges", ligand_atom_range,
    "--ligand_3letter_code", ligand_3letter_code, "--ligand_chain", ligand_chain,"--tip_atom_residues_3letter", *tip_atom_residues_3letter,
    ]

### ADD CONDITIONAL FLAGS / OPTIONS ###
if DO_NOT_pass_tip_atom_residues_flag: # Do NOT pass tip residues to identifier
    cmd_parts.append("--DO_NOT_pass_tip_atom_residues_3letter_to_help_identifier")

if keep_temp_smiles_flag: # Keep SMILES + residue PDB temp files
    cmd_parts.append("--KEEP_temp_smiles_and_res_pdbs_for_debug")

if keep_temp_residue_ligand_files_flag: # Keep residue + ligand TEMP PDBs
    cmd_parts.append("--KEEP_temp_residue_and_temp_ligand_files_for_debug")

if ligand_atom_for_proximity_flag is not None: # Proximity-based GLU/ASP OE/OD renaming
    cmd_parts.extend(["--ligand_atom_for_close_proximity_to_OE2glu_and_OD2asp", ligand_atom_for_proximity_flag,])

if apptainer_path_override is not None: # Optional overrides for container (only if your script argparse includes these)
    cmd_parts.extend(["--apptainer", apptainer_path_override])

if obabel_path_override is not None: # Optional overrides for obabel (only if your script argparse includes these)
    cmd_parts.extend(["--obabel_path", obabel_path_override])

### MAKE FINAL STRING & PRINT ###
command = " ".join(cmd_parts)
print(command)

Execute the above command in the terminal! It is verified to work for the provided example. If things go funky for your theozyme, try the following advice:
- POST-SCRIPT CLARITY DEBUG V1: If stuff is not correctly aligned in outputs, consider changing standard residue chi angles in the STEP_2 helper (build_full_residue_from_subset).
- POST-SCRIPT CLARITY DEBUG V2: If the backbone of a residue is missing, consider adjusting the threshold in 'is_reasonable_pairing' in STEP_3 (superimpose_ideal_residue_on_subset). 

The theozyme PDB can be found at `{theozymes_dir}from_quantum_chemistry/pdb_theozymes/ZETA_2__theozyme_optTS_geom__lig_SZD_artificialBB_theozyme.pdb`. I copied this pdb into the main `from_quantum_chemistry` folder under the name of `ZETA_2__theozyme_optTS_geom_HHHE__lig_SZD_artificialBB.pdb`. That completes this step!

# **II. Theozyme Post-Processing & Preparation for Scaffold Generation** 

### II.A Add ORI Tokens to Theozymes Specifying Desired Protein Center-of-Mass

**NOTE:** Only relevant if using the ORI-token flag in RFdiffusion3 at some point. Otherwise this can be ignored.

The first thing we need to do is get a coordinate for where we want to place our center-of-mass. This coordinate can also serve as the center of a sphere of multiple ORI tokens (i.e., we create many different center-of-mass inputs around a fixed point).


STEP 1: To execute this, we need to take a theozyme PDB file and then install a pseudo ORI token into it by entering this command in the pymol terminal:

*cmd.delete("molecule1");cmd.pseudoatom(object="molecule1", pos=[-1,3,2], elem="ORI", name="ORI", vdw=1.5, hetatm=True, chain='z', segi='z', resn="ORI"); cmd.show("sphere", "molecule1");*

**NOTE:** You may need to zoom out, move the clipping plane way out of the screen, and show molecule 1 as a sphere ("S" -> "spheres") to see the ORI token pop up in pymol. Sometimes it spawns in far away from what you are looking at. 

STEP 2: Then you can click the 'Action Button' on molecule1 -> 'drag coordinates' and physically drag the ORI token to a candidate spot for the center of a sampling sphere. To get the coordinates of the new ORI token for the input below, simply run this command in the pymol terminal:

*iterate_state 1, molecule1, print(name, x, y, z)* 


**NOTE:** You must do this in the 'drag coords' mode, do not hit 'done'.

STEP 3: Once you have the coordinates, paste them into the cell below which will generate a command to run a script which will install the ORI token into your PDB. Alternatively, the script can install a sphere of ORI tokens around that point.

For this example, it is reasonable to assume that the Zn(II) atom could be a good starting point for our center-of-mass since we hypothesize it should be buried in a nice cleft. Thus, we perform the above procedure to move the ORI token near the Zn(II) atom and extract its coordinates for the script below. Try generating just a single PDB with the ORI token at the location we dragged it to, and then try changing the cell below to make a sphere of coordinates around that point. Run the commands that are generated in the below script in your terminal.

#### (Option 1 Worked Example with the Zinc Protease from the PDB)

In [ ]:
############################
### ORI TOKEN GENERATION ###
############################

# !!! OPTION 1 WORKED EXAMPLE WITH THE ZINC PROTEASE FROM THE PDB !!! # 

### INPUT PARAMETERS ###
input_pdb    = f'{theozymes_dir}from_PDB_structure/step2__splitting/pdb_00001qji__theozyme_HEHHYM__lig_TSA.pdb'
output_dir   = f'{theozymes_dir}from_PDB_structure/step3__add_ori_tokens/'
center_coord = '18.012922286987305 24.915010452270508 21.94306182861328'

# Optional alt example:
# center_coord = '-5.401254653930664 -0.7388721704483032 16.641462326049805'

### SPHERE OPTIONS ###
generate_sphere = False           # False → single ORI at center
sphere_mode     = "volume"        # "surface" or "volume" (only used if generate_sphere=True)
radius          = 3               # Å (only used if generate_sphere=True)
num_ORI         = 25              # total ORIs if sphere=True (exact count, includes center)

### OTHER OPTIONAL FLAGS (ADVANCED CONTROLS) ###
specify_ori_token_specific_pdb_properties = False
chain        = "X"                # chain to contain ORI tokens | default = chain X
serial_start = 1                  # indexing to start ORI token atom numbering | default = 999 (NOTE: This must always be greater than the last HETATM atom numbering)
resseq_start = 1                  # indexing to start ORI token residue numbering | default = 1
verbose = True

### CONSTANTS ###
script = f"{scripts_dir}add_ORI_token_to_PDB.py"

### GENERATE COMMAND ###
command = (f"python {script} "
           f"--input_pdb {input_pdb} "
           f"--output_dir {output_dir} "
           f'--center "{center_coord}" ')

if specify_ori_token_specific_pdb_properties:
    command += (f"--chain {chain} "
                f"--serial_start {serial_start} "
                f"--resseq_start {resseq_start} ")
if generate_sphere:
    command += (f"--sphere "
                f"--mode {sphere_mode} "
                f"--radius {radius} "
                f"--sampling_size {num_ORI} ")
if verbose:
    command += "--verbose "

### PRINT COMMAND ###
print(command)

Repeat this step for all of the PDB files from the step 2 splitting stuff we performed in Section I.I.B. Then open up some of the PDB files in pymol to validate that the ORI token placements look good, especially the sphere of coordinates. When viewing files, you can use the following command to make the spheres larger or smaller: \
\
*set sphere_scale, {num_value}* where *num_value* = any positive real number (e.g., *set sphere_scale, 0.5*) 

#### (Option 2 Worked Example with the Zinc Esterase from Quantum Chemistry)

In [ ]:
############################
### ORI TOKEN GENERATION ###
############################

# !!! OPTION 2 WORKED EXAMPLE WITH THE ZINC ESTERASE FROM QUANTUM CHEMISTRY !!! # 

### INPUT PARAMETERS ###
input_pdb    = f'{theozymes_dir}from_quantum_chemistry/ZETA_2__theozyme_optTS_geom_HHHE__lig_SZD_artificialBB.pdb'
output_dir   = f'{theozymes_dir}from_quantum_chemistry/stepFINAL__add_ori_tokens/'
center_coord = '-0.7381851077079773 -1.2144893407821655 -0.3365115821361542'

# Optional alt example:
# center_coord = '-5.401254653930664 -0.7388721704483032 16.641462326049805'

### SPHERE OPTIONS ###
generate_sphere = True            # False → single ORI at center
sphere_mode     = "volume"        # "surface" or "volume" (only used if generate_sphere=True)
radius          = 2               # Å (only used if generate_sphere=True)
num_ORI         = 15              # total ORIs if sphere=True (exact count, includes center)

### OTHER OPTIONAL FLAGS (ADVANCED CONTROLS) ###
specify_ori_token_specific_pdb_properties = False
chain        = "X"                # chain to contain ORI tokens | default = chain X
serial_start = 1                  # indexing to start ORI token atom numbering | default = 999 (NOTE: This must always be greater than the last HETATM atom numbering)
resseq_start = 1                  # indexing to start ORI token residue numbering | default = 1
verbose = True

### CONSTANTS ###
script = f"{scripts_dir}add_ORI_token_to_PDB.py"

### GENERATE COMMAND ###
command = (f"python {script} "
           f"--input_pdb {input_pdb} "
           f"--output_dir {output_dir} "
           f'--center "{center_coord}" ')

if specify_ori_token_specific_pdb_properties:
    command += (f"--chain {chain} "
                f"--serial_start {serial_start} "
                f"--resseq_start {resseq_start} ")
if generate_sphere:
    command += (f"--sphere "
                f"--mode {sphere_mode} "
                f"--radius {radius} "
                f"--sampling_size {num_ORI} ")
if verbose:
    command += "--verbose "

### PRINT COMMAND ###
print(command)

Open up some of the PDB files in pymol to validate that the ORI token placements look good, especially the sphere of coordinates. When viewing files, you can use the following command to make the spheres larger or smaller: \
\
*set sphere_scale, {num_value}* where *num_value* = any positive real number (e.g., *set sphere_scale, 0.5*) 

### II.B (OPTIONAL) Copy the RFdiffusion3-Ready Theozymes into a New Folder

At this point we now have everything needed to prepare for running RFdiffusion3! You can feel free to add custom steps (e.g., flip histidine conformation or change coordinating nitrogen from epsilon to delta) in Section I. or Section II. or subtract the optional steps.

At this point, I copied the final theozymes containing the ORI tokens into a new folder called "rfdiffusion3_inputs". I have 27 inputs spanning 3 different theozymes (25 ORI token placements for HEHH, 1 ORI token placement for HEHHY, and 1 ORI token placement for HEHHYM). 

Likewise, for the quantum chemistry-derived theozymes, I copied them into a folder called "rfdiffusion3_inputs" in its respective subdirectory, where I have 11 inputs spanning 1 unique theozyme (11 ORI tokens for HHHE).

### Move to Section III.

# **III. Generate Backbones at Scale**

## III.A Setup RFdiffusion3 Inference

#### What's new in RFdiffusion3 vs RFdiffusion2

RFdiffusion3 is an all-atom diffusion model and introduces several conditioning options that don't exist in RFdiffusion2. The cells below configure them. Quick reference:

- **Dialect 2** (`"dialect": 2`) &mdash; the input-spec format expected by `rfd3 design`. Required key in every JSON. RFdiffusion3 is on dialect 2; older dialects are deprecated.
- **Unindexed residues** (`unindexed_set_id`) &mdash; catalytic residues whose sequence position is *not* pre-specified. RFdiffusion3 places them anywhere along the backbone during inference, dramatically expanding the search space vs. fixed-position scaffolding.
- **Fixed atoms** (`fixed_atoms_set_id`) &mdash; for each unindexed residue, the subset of side-chain atoms whose coordinates are held constant during diffusion (e.g. the imidazole ring atoms of a Zn-coordinating histidine).
- **Atomwise RASA conditioning** (`atomwise_rasa_set_id`) &mdash; per-atom *Relative Accessible Surface Area* targets on ligand atoms (`select_buried` / `select_partially_buried` / `select_exposed`). Steers RFd3 to produce scaffolds where specified ligand atoms are e.g. buried in the active site or solvent-exposed.
- **Atomwise H-bond conditioning** (`atomwise_hbond_set_id`) &mdash; explicit donor/acceptor specifications between specific ligand atoms and protein residues. Use this to design catalytic H-bond networks (oxyanion holes, general-base activation, etc.). Note that the *training* of this feature requires HBPLUS, but inference only requires foundry to be installed with the rfd3 extra.
- **Classifier-free guidance (CFG)** (`use_classifier_free_guidance`, `cfg_scale`) &mdash; at inference time, samples from both the conditioned and unconditioned models and extrapolates between them by `cfg_scale`. Higher values push designs to satisfy the conditioning more aggressively at the cost of structural diversity.

For full details see the RFdiffusion3 preprint ([Butcher et al., *bioRxiv* 2025](https://doi.org/10.1101/2025.09.18.676967)) and the foundry RFD3 docs at [`Software/foundry/models/rfd3/README.md`](../Software/foundry/models/rfd3/README.md).

The 1QJI active site is cropped at three levels of context. Each becomes one group in the JSON generator below:

**HEHH** &mdash; `A92`, `A93`, `A96`, `A102` (His, Glu, His, His): the three zinc-ligating histidines plus the general-base glutamate. \
**HEHHY** &mdash; the above `+ A149` (Tyr). \
**HEHHYM** &mdash; the above `+ A147`, `A149` (Met, Tyr).

The worked example below runs **HEHH**; configs are generated for all three so you can compare.

In [ ]:
#####################################################################
### CELL TO HELP GENERATE THE FIXED ATOM DEFS FOR THE NEXT CELL   ###
#####################################################################
# List the side-chain atoms to hold fixed per catalytic residue, and this prints
# `fixed_atoms_defs` and `unindexed_defs` blocks ready to paste into the next
# cell. "ALL" expands to every side-chain heavy atom present in the reference.

### REFERENCE PDB (expands "ALL" and checks the residue identities) ###
reference_pdb = f"{theozymes_dir}from_PDB_structure/step2__splitting/pdb_00001qji__theozyme_HEHHYM__lig_TSA.pdb"

### SPECIFY ATOMS FOR FIXED RESIDUES, PER THEOZYME VARIANT ###
_HIS = ["NE2", "CD2", "CG", "CB", "ND1", "CE1"]
residue_atoms_by_set = {
    ### SET 1 -- HEHH ###
    1: {
        "A92":  _HIS,                          # HIS - zinc ligand
        "A93":  ["OE1", "OE2", "CD", "CG"],    # GLU - general base
        "A96":  _HIS,                          # HIS - zinc ligand
        "A102": _HIS,                          # HIS - zinc ligand
    },
    ### SET 2 -- HEHHY (adds Tyr149) ###
    2: {
        "A92":  _HIS,
        "A93":  ["OE1", "OE2", "CD", "CG"],
        "A96":  _HIS,
        "A102": _HIS,
        "A149": ["ALL"],                       # TYR
    },
    ### SET 3 -- HEHHYM (adds Met147, Tyr149) ###
    3: {
        "A92":  _HIS,
        "A93":  ["OE1", "OE2", "CD", "CG"],
        "A96":  _HIS,
        "A102": _HIS,
        "A147": ["SD", "CE", "CG"],            # MET
        "A149": ["ALL"],                       # TYR
    },
}

### BUILD IT ###
BACKBONE = {"N", "CA", "C", "O", "OXT"}
ref_atoms, ref_name = {}, {}
with open(reference_pdb) as fh:
    for line in fh:
        if not line.startswith("ATOM"):
            continue
        key, atom = f"{line[21]}{int(line[22:26])}", line[12:16].strip()
        ref_name.setdefault(key, line[17:20].strip())
        if atom not in BACKBONE and not atom.startswith("H"):
            ref_atoms.setdefault(key, []).append(atom)

### PRINT ###
print("### PASTE INTO fixed_atoms_defs IN THE NEXT CELL ###\n")
print("fixed_atoms_defs = {")
for set_id, spec in residue_atoms_by_set.items():
    print(f"    {set_id}: {{")
    for res, atoms in spec.items():
        present = ref_atoms.get(res, [])
        atoms   = present if atoms == ["ALL"] else atoms
        missing = [a for a in atoms if a not in present]
        note    = ref_name.get(res, "NOT IN REFERENCE")
        if missing:
            note += f" -- atoms absent from reference: {','.join(missing)}"
        print(f'        "{res}": "{",".join(atoms)}",'.ljust(56) + f"# {note}")
    print("    },")
print("}")

print("\n### PASTE INTO unindexed_defs IN THE NEXT CELL ###\n")
print("unindexed_defs = {")
for set_id, spec in residue_atoms_by_set.items():
    print(f'    {set_id}: "{",".join(spec)}",')
print("}")

### Generate JSON Inputs

**Generate JSON Inputs for Input PDBs**

This cell automates the creation of RFdiffusion3‐compatible JSON configuration files from a directory of PDBs.  Each JSON can encode:

- **Multiple parameter combinations** (“combos”) per PDB, if specified. Will create all possible combos based on lists given below.  
- **Multiple ORI tokens** per PDB, if present.  

---

**A) Grouping Definitions**

You define one or more **groups** keyed by a file‐name prefix.  Each group dictionary can include:

```python
grouping_info_by_prefix_andOR_substring = {
    "group1": {
        # ── Optional behavior ────────────────────────────────────────────────────────
        "suffix_to_add":          "SUFFIX",     
          • Appended to every JSON key *and* filename, e.g. `pdb1_combo1SUFFIX.json`
        "pdb_substring_grouping": "0_1_1",      
          • Further restricts matching to files containing this substring:
            `group1*0_1_1*.pdb`

        # ── Optional ligan inputs ───────────────────────────────────────────────────
        "ligand": "SZA",                        
          • Ligand identifier(s).  Must be a single string but can be a list (eg., "SZA,ZN").

        # ── Optional “set_id” parameters ────────────────────────────────────────────
        #   each may be an `int` or list `[int,...]` to generate multiple combos:
        "fixed_atoms_set_id":    [1,2,3],       
          • Picks which atom‐fixing definition to use (see `fixed_atoms_defs`)
        "unindexed_set_id":      1,             
          • Picks which residue‐unindexing string to use (see `unindexed_defs`)
        "contig_set_id":         1,             
          • Picks which contiguous segment definitions to use (see `contig_defs`)
        "length_set_id":         1,             
          • Picks which sequence‐length window to use (see `length_defs`)
        "unfix_sequence_set_id": 1,             
          • Picks which side‐chains to remove from motif (see `unfix_sequence_defs`)
        "atomwise_rasa_set_id":  1,             
          • Picks which per‐atom RASA map to include (see `atomwise_rasa_defs`)
        "atomwise_hbond_set_id": 1,             
          • Picks which per‐atom H‐bond map to include (see `atomwise_hbond_defs`)

        # ── Optional boolean flags ──────────────────────────────────────────────────
        "redesign_motif_sidechains": [True, False],
          • Whether to allow redesign of the motif side‐chains, this shows you would do combinations w True & False, a json for each
        # ── Overrides ───────────────────────────────────────────────────────────────
        "out_path":        "{base}_combo{combo_idx}.json",
          • Custom filename template; available formatting keys:
              {base}, {combo_idx}, {suffix}
    },
}


In [ ]:
##############################################
### 1) GENERATE JSON INPUTS FOR INPUT PDBS ###
##############################################

lig = "TSA"          # the phosphonamidate transition-state analog cropped from PDB 1QJI

print_pdbs_processed = False

### INPUT PDB DIRECTORY (WITH ORI TOKENS IF YOU WANT THEM) ###
input_root    = f"{theozymes_dir}from_PDB_structure/rfdiffusion3_inputs/"

### INFERENCE CONTROLS VIA GROUPINGS ###
# One group per theozyme variant. Keys are matched as a FILENAME PREFIX, and
# the trailing "__" matters: without it "HEHH" would also match HEHHY/HEHHYM.
_COMMON = {
    "pdb_substring_grouping":    f"{lig}",
    "suffix_to_add":             "",
    "dialect":                    2,
    "redesign_motif_sidechains":  False,
    "ligand":                     f"{lig}",
    "length_set_id":              [1, 2],   # 120-150 and 170-190
    "contig_set_id":              None,
    "unfix_sequence_set_id":      None,
    "ori_set_id":                 None,     # ORI tokens are read from each input PDB
    "atomwise_rasa_set_id":       None,     # see atomwise_rasa_defs below to enable
    "atomwise_hbond_set_id":      None,
}
grouping_info_by_prefix_andOR_substring = {
    "pdb_00001qji__theozyme_HEHH__":   {**_COMMON, "unindexed_set_id": 1, "fixed_atoms_set_id": 1},
    "pdb_00001qji__theozyme_HEHHY__":  {**_COMMON, "unindexed_set_id": 2, "fixed_atoms_set_id": 2},
    "pdb_00001qji__theozyme_HEHHYM__": {**_COMMON, "unindexed_set_id": 3, "fixed_atoms_set_id": 3},
}

# Side-chain atoms held fixed during diffusion: the tip atoms that make the
# catalysis. CA is kept as a placeholder by RFdiffusion3 regardless.
_HIS_TIP = "NE2,CD2,CG,CB,ND1,CE1"
_GLU_TIP = "OE1,OE2,CD,CG"
_TYR_TIP = "OH,CZ,CE1,CD1,CE2,CD2,CG,CB"
_MET_TIP = "SD,CE,CG"
fixed_atoms_defs = {
    1: {"A92": _HIS_TIP, "A93": _GLU_TIP, "A96": _HIS_TIP, "A102": _HIS_TIP},
    2: {"A92": _HIS_TIP, "A93": _GLU_TIP, "A96": _HIS_TIP, "A102": _HIS_TIP,
        "A149": _TYR_TIP},
    3: {"A92": _HIS_TIP, "A93": _GLU_TIP, "A96": _HIS_TIP, "A102": _HIS_TIP,
        "A147": _MET_TIP, "A149": _TYR_TIP},
}

# Catalytic residues, by theozyme variant (1QJI numbering).
unindexed_defs = {
    1: "A92,A93,A96,A102",                 # HEHH   - His92, Glu93, His96, His102
    2: "A92,A93,A96,A102,A149",            # HEHHY  - adds Tyr149
    3: "A92,A93,A96,A102,A147,A149",       # HEHHYM - adds Met147, Tyr149
}

unfix_sequence_defs = {}

contig_defs = {}

### GLOBAL PROPERTY SET IDs ###
length_defs = {
    1: "120-150",
    2: "170-190",
}

# ORI tokens come from the input PDBs themselves (one per ORI_xx file), so this
# table is unused; ori_set_id is None above.
ori_defs = {}

### ATOM CONDITIONING SET IDs ###
# Atomwise RASA conditioning is OFF by default (atomwise_rasa_set_id = None),
# which is what the configs shipped in rfd3_json/ use. To enable it, classify the
# TSA atoms yourself by how buried you want them and set atomwise_rasa_set_id.
# TSA atom names are ZN1, P1, N1-N6, O1-O9, C1-C36 (see the input PDB).
atomwise_rasa_defs = {
    # 1: {"select_buried":           {f"{lig}": "ZN1,P1,..."},
    #     "select_partially_buried": {f"{lig}": "..."},
    #     "select_exposed":          {f"{lig}": "..."}},
}

# Atomwise H-bond conditioning, off by default. See the RFdiffusion3 docs for
# the donor/acceptor syntax.
atomwise_hbond_defs = {}

### WHERE TO DUMP JSONS ###
rfd3_json_dir = rfd3_json_dir
os.makedirs(rfd3_json_dir, exist_ok=True)

### FUNCTION ###
def to_list(x):
    return x if isinstance(x, (list, tuple)) else [x]

### MAKE THE JSON FILES ###
print(f"{rfd3_json_dir}")
overall_total_pdbs = 0
overall_matched    = 0
overall_jsons      = 0

for prefix, cfg in grouping_info_by_prefix_andOR_substring.items():
    print(f"\n=== Group '{prefix}' ===")
    substr    = cfg.get("pdb_substring_grouping", "")
    pattern   = f"{prefix}*{substr}*.pdb" if substr else f"{prefix}*.pdb" # pattern   = f"{prefix}*{substr}" if substr else f"{prefix}*.pdb"
    pdb_paths = sorted(glob.glob(os.path.join(input_root, pattern)))
    group_total = len(pdb_paths)
    overall_total_pdbs += group_total
    print(f"Found {group_total} PDBs matching '{pattern}'")

    # build combos once --> build it dynamically:
    #  1) collect all the “*_set_id” keys
    set_id_keys = sorted(k for k in cfg.keys() if k.endswith("_set_id"))
    #  2) collect the boolean flags (if present)
    flag_order = ["redesign_motif_sidechains"]
    flag_keys  = [k for k in flag_order if k in cfg]
    #  3) concatenate
    param_names = set_id_keys + flag_keys
    param_values = [to_list(cfg.get(name)) for name in param_names]
    combos       = list(itertools.product(*param_values))
    combo_count  = len(combos)
    print(f"Will generate {combo_count} JSON combos per PDB")

    group_matched = 0
    group_jsons   = 0

    for pdb_path in pdb_paths:
        base = os.path.splitext(os.path.basename(pdb_path))[0]

        # determine ORI_tokens either from cfg or by parsing the PDB
        ori_id = cfg.get("ori_set_id")
        if ori_id is not None:
            ORI_tokens = ori_defs[ori_id]
        else:
            # fall back to parsing every HETATM … ORI line
            ORI_tokens = []
            with open(pdb_path) as f:
                for line in f:
                    if line.startswith("HETATM") and " ORI " in line:
                        cols = line.split()
                        ORI_tokens.append([float(cols[6]), float(cols[7]), float(cols[8])])
                        
        multi_ori = len(ORI_tokens) > 1
        group_matched += 1
        overall_matched += 1
        if print_pdbs_processed:
            print(f" - '{base}': {len(ORI_tokens)} ORI token(s)")

        suffix = cfg.get("suffix_to_add", "")
        for combo_idx, combo in enumerate(combos, start=1):
            combo_dict = dict(zip(param_names, combo))
            json_dict  = {}
            # if there are no ORI tokens, we still want one pass, but without adding the field
            ori_iter = ORI_tokens if ORI_tokens else [None]

            for idx, ori in enumerate(ori_iter, start=1):
                entry = {"input": pdb_path}
                ### OVERRIDES ###
                if "dialect" in cfg:
                    entry["dialect"] = cfg["dialect"]
                # boolean flags
                for flag in ("redesign_motif_sidechains",): # ADD MORE BOOLEAN FLAGS HERE
                    val = combo_dict[flag]
                    if val is not None:
                        entry[flag] = val
                ### MOTIF SCAFFOLDING ###
                if "ligand" in cfg:
                    entry["ligand"] = to_list(cfg["ligand"])[0]
                if combo_dict["length_set_id"] is not None:
                    entry["length"] = length_defs[combo_dict["length_set_id"]]
                if combo_dict["contig_set_id"] is not None:
                    entry["contig"] = contig_defs[combo_dict["contig_set_id"]]
                if combo_dict["unindexed_set_id"] is not None:
                    entry["unindex"] = unindexed_defs[combo_dict["unindexed_set_id"]]
                if combo_dict["unfix_sequence_set_id"] is not None:
                    entry["select_unfix_sequence"] = unfix_sequence_defs[combo_dict["unfix_sequence_set_id"]]
                if combo_dict["fixed_atoms_set_id"] is not None:
                    entry["select_fixed_atoms"] = fixed_atoms_defs[combo_dict["fixed_atoms_set_id"]]
                if combo_dict["atomwise_rasa_set_id"] is not None:
                    entry.update(atomwise_rasa_defs[combo_dict["atomwise_rasa_set_id"]])
                if combo_dict["atomwise_hbond_set_id"] is not None:
                    entry["atomwise_hbond"] = atomwise_hbond_defs[combo_dict["atomwise_hbond_set_id"]]
                ### ORI TOKEN INSERTION ###
                if ori is not None:
                    entry["ori_token"] = ori
                ### NAMING ###
                if suffix:
                    suffix_key = suffix + "_"
                else:
                    suffix_key = ""
                if multi_ori:
                    key = f"ORI_{idx}_i"
                else:
                    key = f"i"
                json_dict[key] = entry

            # write JSON
            if multi_ori:
                default_out = os.path.join(rfd3_json_dir, f"{base}_C{combo_idx}{suffix_key}.json")
            else:
                default_out = os.path.join(rfd3_json_dir, f"{base}_C{combo_idx}{suffix_key}.json")
            json_file = os.path.join(default_out)
            with open(json_file, "w") as fo:
                json.dump(json_dict, fo, indent=2) 
            group_jsons += 1
            overall_jsons += 1

    ### PRINT GROUP SUMMARY ###
    print(f"Group '{prefix}' summary:")
    print(f"  PDBs processed:     {group_matched}/{group_total}")
    print(f"  Combos per PDB:     {combo_count}")
    print(f"  JSON files created: {group_jsons}")
    print("-" * 60)

### PRINT OVERALL SUMMARY ###
print("\n=== Overall Summary ===")
print(f"Total PDBs found:           {overall_total_pdbs}")
print(f"Total PDBs with ORI tokens: {overall_matched}")
print(f"Total JSON files generated: {overall_jsons}")

### Production Run (with Optional Hyperparameter Sweeping)

#### RFd3 sampler hyperparameters (set in the next cell)

The RFd3 inference sampler exposes several knobs. Sensible starting values are shown in the cell below; treat them as a hyperparameter sweep when in doubt &mdash; the cell builds the Cartesian product of all `*_list` variables.

- **`step_scale`** &mdash; multiplier on the per-step update magnitude. Larger values produce more diverse but noisier outputs; smaller values produce smoother, more conservative trajectories. *Typical: 1.0&ndash;2.0.*
- **`gamma_0`** &mdash; initial noise scale at the start of the trajectory (high-noise end of the schedule). *Typical: 0.4&ndash;0.8.*
- **`gamma_min`** &mdash; minimum noise scale at the end of the trajectory (low-noise end of the schedule). *Typical: 0.05&ndash;0.2.*
- **`s_jitter_origin`** &mdash; jitter added to the ORI-token position each step (&Aring;ngstr&ouml;ms), to encourage exploration around the specified center-of-mass. *Typical: 0.5&ndash;2.0.*
- **`cfg_scale`** &mdash; classifier-free guidance strength (only used when `use_classifier_free_guidance=True`). *Typical: 1.0&ndash;2.0; >2 risks over-constrained, low-diversity designs.*
- **`diffusion_batch_size`** &times; **`n_batches`** &mdash; total designs per JSON config = `diffusion_batch_size * n_batches`. The split affects GPU memory; the total is what matters scientifically.

See [Butcher et al., *bioRxiv* 2025](https://doi.org/10.1101/2025.09.18.676967) for the full sampler equations.

In [ ]:
##############################################
### BUILD COMMAND LINES FOR RFD3 INFERENCE ###
##############################################

json_filters = [
    "theozyme_HEHH__",
]  # e.g. ["group1","group2"], or [] to include all
# The trailing "__" keeps this to the HEHH configs; without it HEHHY and
# HEHHYM would match too.

### INPUTS ###
base_output_dir      = f"{rfdiffusion3_out_dir}i1/"

### INFERENCE PARAMETERS (single–value defaults, but we’ll override) ###
diffusion_batch_size         = 4
n_batches                    = 6
dump_trajectories            = False
cleanup_virtual_atoms        = True

# ── 1) the lists you want to sweep ─────────────────────────────────────────
use_classifier_free_guidance_list = [True]
cfg_scale_list                    = [1.5]
step_scale_list                   = [1.5]
gamma_0_list                      = [0.6]
gamma_min_list                    = [0.1]
s_jitter_origin_list              = [1.5]

### DIRECTORY TO PARSE FOR JSONS & COMMANDS NAMING ###
json_dir      = f"{rfd3_json_dir}"
commands_name = "RFdiffusion3_production"

### RFD3 INFERENCE CLI ###
# Commands invoke the `rfd3 design` CLI from the foundry package (Software/foundry,
# branch=production). Setup once before running:
#
#   # Editable install of the bundled submodule (recommended -- pinned to the tested commit):
#   pip install -e "Software/foundry[rfd3]"
#   # OR install from PyPI:
#   pip install "rc-foundry[rfd3]"
#
#   # Download the checkpoint (defaults to ~/.foundry/checkpoints):
#   foundry install rfd3 --checkpoint-dir <path/to/ckpt/dir>
#
# Foundry auto-discovers checkpoints from ~/.foundry/checkpoints plus any colon-separated
# directories listed in the FOUNDRY_CHECKPOINT_DIRS env var, so an explicit `ckpt_path` is
# usually unnecessary. See Software/foundry/models/rfd3/README.md for full docs.
checkpoint_path = None    # e.g. os.path.expanduser("~/.foundry/checkpoints/rfd3_latest.ckpt")
os.makedirs(base_output_dir, exist_ok=True)
if json_filters:
    json_files = sorted(set(m for substr in json_filters for m in glob.glob(os.path.join(json_dir, f"*{substr}*.json"))))
else:
    json_files = sorted(glob.glob(os.path.join(json_dir, "*.json")))
print(f"Found {len(json_files)} JSON file(s) matching filters: {json_filters}")

commands_file = os.path.join(cmds_dir, commands_name); command_count = 0
with open(commands_file, "w") as fh:
    for cfg_path in sorted(json_files):
        for use_classifier_free_guidance in use_classifier_free_guidance_list:
            # only sweep cfg_scale when guidance is on
            cfg_scales = cfg_scale_list if use_classifier_free_guidance else [None]
            for cfg_scale in cfg_scales:
                for step_scale, gamma_0, gamma_min, s_jitter_origin in product(step_scale_list, gamma_0_list, gamma_min_list, s_jitter_origin_list):
                    seed = random.randint(0, 2**32 - 1)
                    # build a unique out_dir
                    out_dir = f"{base_output_dir}cfg_{'T' if use_classifier_free_guidance else 'F'}__cfgsc_{format(cfg_scale, '.2f').replace('.', '_') if cfg_scale is not None else 'NA'}__step_{format(step_scale, '.2f').replace('.', '_')}__gam0_{format(gamma_0, '.2f').replace('.', '_')}__gamMIN_{format(gamma_min, '.2f').replace('.', '_')}__jit_{format(s_jitter_origin, '.2f').replace('.', '_')}"
                    os.makedirs(out_dir, exist_ok=True)
                    cmd = (
                        f"rfd3 design inputs={cfg_path} out_dir={out_dir} "
                        + (f"ckpt_path={checkpoint_path} " if checkpoint_path is not None else "")
                        + f"dump_trajectories={dump_trajectories} diffusion_batch_size={diffusion_batch_size} n_batches={n_batches} cleanup_virtual_atoms={cleanup_virtual_atoms} "
                        + f"inference_sampler.s_jitter_origin={s_jitter_origin} inference_sampler.use_classifier_free_guidance={use_classifier_free_guidance} "
                        + (f"inference_sampler.cfg_scale={cfg_scale} " if cfg_scale is not None else "")
                        + f"inference_sampler.step_scale={step_scale} inference_sampler.gamma_0={gamma_0} inference_sampler.gamma_min={gamma_min} seed={seed} "
                        + f"skip_existing=False prevalidate_inputs=True"
                    )
                    fh.write(cmd + "\n")
                    command_count += 1
            
### SETUP BATCH JOBS ###
qtime        = '06:30:00'
cmds_per_job = 15
cores        = '1'
memory       = '16g'
queue        = 'gpu'
job_name     = os.path.basename(commands_file)
submit_file  = f'{slurm_submit_dir}{job_name}.sh'
num_jobs     = math.ceil(command_count / cmds_per_job)

print(f"\nStructures per Cmd =", diffusion_batch_size*n_batches, f"\nTotal Structures to Generate =", diffusion_batch_size*n_batches*command_count)
print(f"\nNumber of Cmds =", command_count, f"\nNumber of Jobs =", num_jobs)
print("Job Name =", job_name)
print("\nCommands File for Testing:", f"\n{commands_file}")
print(f"\nNavigate to Output Directory:", f"\ncd {base_output_dir}\n")
functions.submit_array_job(commands_file, qtime, cores, job_name,memory, submit_file, logs_dir,num_jobs+1, cmds_per_job, queue)

## III.B Rapid Output Quality Filtering (Using RFd3 Jsons)

### Parse JSONs into Combined File

In [ ]:
############################################
### COMBINE JSON FILES INTO SINGLE JSON  ###
############################################

### INPUT DIRECTORY TO PARSE ###
rfd3_output_dir_to_parse = f"{rfdiffusion3_out_dir}i1/"

### OUTPUT JSON PATH ###
output_combined_json_path = f"{rfdiffusion3_out_dir}i1/combined_stats.json"

### PARSING & SORTING LOGIC ###
json_files = glob.glob(f'{rfd3_output_dir_to_parse}**/*.json', recursive=True) # sort alphanumerically by absolute path
json_files = sorted(json_files, key=lambda p: os.path.abspath(p)) # skip our own combined_stats.json if it lives in the same folder
json_files = [ p for p in json_files # skip our own combined_stats.json if it lives in the same folder
    if os.path.abspath(p) != os.path.abspath(output_combined_json_path)
]
print(f"Found {len(json_files)} JSON files to parse")

all_records = []
for path in json_files:
    # load the JSON
    with open(path, 'r') as f:
        rec = json.load(f)

    # compute our metadata
    abs_path   = os.path.abspath(path)
    cifgz_path = abs_path.replace('.json', '.cif.gz')
    subdir     = os.path.basename(os.path.dirname(path))

    # build a new dict so that our three keys come first
    ordered_rec = {
        'rfd3_json_path':  abs_path,
        'rfd3_cifgz_path': cifgz_path,
        'subdirectory':    subdir,
    }

    # ensure rec is a dict before merging
    if isinstance(rec, dict):
        ordered_rec.update(rec)
        ### ADD: MULTIPLE NORMALIZED Rg METRICS USING NESTED METRICS #
        metrics_dict = ordered_rec.get("metrics")
        if isinstance(metrics_dict, dict):
            Rg = metrics_dict.get("radius_of_gyration", None)
            N  = metrics_dict.get("num_residues", None)
            try:
                Rg = float(Rg)
                N  = float(N)
            except (TypeError, ValueError):
                Rg = None
                N  = None
            if Rg is not None and N is not None and N > 0:
                # 1) Globular-like scaling: Rg / N^(1/3)
                metrics_dict["radius_of_gyration.norm_by_globularity"] = Rg / (N ** (1.0 / 3.0))

                # 2) Coil-like scaling: Rg / N^0.58  (approx. random-coil exponent)
                metrics_dict["radius_of_gyration.norm_by_coil"] = Rg / (N ** 0.58)

                # 3) Ideal-sphere normalization (dimensionless sphericity factor)
                v_res = 110.0      # Å^3 per amino acid (average)
                conv  = 1.212e-3   # Å^3 -> nm^3 (incl. packing/hydration fudge)
                V_nm3 = N * v_res * conv
                R_sphere_nm = (3.0 * V_nm3 / (4.0 * np.pi)) ** (1.0 / 3.0)
                R_sphere_A  = R_sphere_nm * 10.0  # back to Å
                #metrics_dict["radius_of_gyration.ideal_sphere"] = R_sphere_A
                metrics_dict["radius_of_gyration.norm_by_ideal_sphere"] = (Rg / R_sphere_A if R_sphere_A > 0.0 else None)
            # write back (not strictly necessary since it's same object, but explicit)
            ordered_rec["metrics"] = metrics_dict
        ##############################################################
        all_records.append(ordered_rec)
    else:
        print(f"⚠️  Skipping {path!r}: top‐level JSON is not an object")

# write combined JSON
with open(output_combined_json_path, 'w') as out:
    json.dump(all_records, out, indent=2)
print(f"Saved {len(all_records)} records → {output_combined_json_path}")

### OPTIONAL: Filter & Investigate Hyperparameter Sweep

In [ ]:
###########################################
### LOAD COMBINED JSON & FILTER METRICS ###
###########################################

### COMBINED JSON PATH FROM ABOVE ###
output_combined_json_path = f"{rfdiffusion3_out_dir}i1/combined_stats.json"

# The combined statistics file is written after RFdiffusion3 inference and the
# JSON-merge step above. Run those first; this cell reads their output.
if not os.path.exists(output_combined_json_path) or not json.load(open(output_combined_json_path)):
    raise FileNotFoundError(
        f"No RFdiffusion3 statistics found at {output_combined_json_path}.\n"
        f"  Run the inference commands emitted above, then the JSON-merge cell,\n"
        f"  before running this filtering step."
    )


### HELPER TO BUILD EXTREME FUNCTIONS ###
def take_json_nest_min_or_max(prefix: str, agg: str = 'max'):
    def fn(df: pd.DataFrame) -> pd.Series:
        cols = df.filter(regex=rf'^{prefix}\.').columns
        if agg == 'max':
            return df[cols].max(axis=1)
        elif agg == 'min':
            return df[cols].min(axis=1)
        else:
            raise ValueError("agg must be 'max' or 'min'")
    fn.__name__ = f"{agg}_{prefix.split('.')[-1]}"
    return fn

### FILTERS ###
conditions = [
    ### sidechain quality ###
    (take_json_nest_min_or_max('metrics.join_point_rmsd_by_token', 'max'), '<', 0.8),
    #('metrics.insertion.mae', '<', 0.6), # MAE = (no alignment) | really just for debugging
    ('metrics.insertion.rmcd', '<', 0.6), # RMCD = (centers but no rotation) - should agree w RMSD
    ('metrics.insertion_rmsd', '<', 0.6), # RMSD = (full optimal alignment) - best metric
    ('metrics.join_point_rmsd', '<', 0.6),
    ('metrics.n_conjoined_residues', '<=', 0), # | NOT SURE WHAT THIS MEANS
    ### diversity content ###
    ('metrics.alanine_content', '<', 0.4),
    ('metrics.glycine_content', '<', 0.3),
    #('metrics.num_ss_elements', '<', 9),
    ('metrics.non_loop_fraction', '>', 0.4),
    ('metrics.loop_fraction', '<', 0.6),
    ('metrics.helix_fraction', '>', 0.05),
    ('metrics.sheet_fraction', '>', 0.000),

    ### backbone quality ###
    ('metrics.max_ca_deviation', '<', 4.0), # 3.8 Å between consecutive Cα atoms is ideal
    ('metrics.n_chainbreaks', '<=', 1),
    ('metrics.n_clashing.interresidue_clashes_w_sidechain', '<=', 3), # includes any sc-sc clash, usually resolved by seq design
    ('metrics.n_clashing.interresidue_clashes_w_backbone', '<=', 0),
   # ('metrics.n_clashing.ligand_min_distance', '>', 2), # backbone nearest dist with ligand
   # ('metrics.radius_of_gyration', '<', 18), # NON-NORMALIZED by number of residues (N)
    ('metrics.radius_of_gyration.norm_by_globularity', '<', 2.75), # NORMALIZED (N^(1/3))
    #('metrics.radius_of_gyration.norm_by_coil', '<', 18), # NORMALIZED (N^(0.58))
    ('metrics.radius_of_gyration.norm_by_ideal_sphere', '<', 0.86), # NORMALIZED by ideal protein sphere | Values ~~1 = very compact / sphere-like &&& Values ≫1 = elongated / dumbbell / multi-lobed
]

### SORT ###
number_of_sorted_to_print = 20
metric_to_sort            = "metrics.loop_fraction" # OR FUNCTION --> take_json_nest_min_or_max('metrics.join_point_rmsd_by_token','max')    # e.g. 'metrics.insertion_rmsd', 'metrics.join_point_rmsd', etc.
sort_by_highest_values    = True                         # True → top N highest values; False → bottom N lowest values

### FILTERING LOGIC ###
with open(output_combined_json_path, 'r') as f:
    records = json.load(f)
df    = pd.json_normalize(records, sep='.')
total = len(df)
print(f"Total records: {total}\n")

# prepare aligned printing
labels      = [(col.__name__ if callable(col) else col) for col, *_ in conditions]
label_width = max(len(str(l)) for l in labels)
opval_strs  = [f"{op} {val}" for *_, op, val in conditions]
opval_width = max(len(s) for s in opval_strs)
num_width   = len(str(total))
op_funcs = {'<': operator.lt, '<=': operator.le, '>': operator.gt, '>=': operator.ge}

# Print each filter’s pass count, aligned
for (col, op, val), label, opval in zip(conditions, labels, opval_strs):
    series = col(df) if callable(col) else df[col]
    mask   = op_funcs[op](series, val)
    count  = mask.sum()
    pct    = count / total * 100
    print(f"[{label:<{label_width}} {opval:<{opval_width}} ]:  "
          f"{count:>{num_width}} / {total:<{num_width}}  ({pct:6.3f}%)")

# Combined filter
combined_mask = pd.Series(True, index=df.index)
for col, op, val in conditions:
    series        = col(df) if callable(col) else df[col]
    combined_mask &= op_funcs[op](series, val)

combined_count = combined_mask.sum()
combined_pct   = combined_count / total * 100
print(f"\nPassed All Conditions → {combined_count} / {total} ({combined_pct:.3f}%)")

### GROUP BY SUBDIRECTORY AND COMPUTE PASS PERCENTAGES ###
# Mark which rows passed
df['passed'] = combined_mask
sub_stats = (df.groupby('subdirectory')['passed'].agg(total='size', passed_count='sum').assign(pass_pct=lambda d: d['passed_count'] / d['total'] * 100).sort_values('pass_pct', ascending=False))
print("Subdirectories ranked by filter pass % (top 10):")
print(sub_stats.head(10)[['total', 'passed_count', 'pass_pct']])
rfd3_json_df_filtered = df[combined_mask]

### SORT PASSING STRUCTURES BY A VALUE & PRINT THEIR PATHS ###
df_tmp = (rfd3_json_df_filtered.assign(_sv=lambda d: metric_to_sort(d)).sort_values('_sv', ascending=not sort_by_highest_values)) if callable(metric_to_sort) else rfd3_json_df_filtered.sort_values(metric_to_sort, ascending=not sort_by_highest_values)
col = '_sv' if callable(metric_to_sort) else metric_to_sort

sel = df_tmp.head(number_of_sorted_to_print)
names = [os.path.basename(p) for p in sel['rfd3_cifgz_path']]
svals = [f"{v:.4f}" for v in sel[col]]
w1, w2 = max(map(len, names)), max(map(len, svals))

print(f"\n{'TOP' if sort_by_highest_values else 'BOTTOM'} {number_of_sorted_to_print} "
      f"{'highest' if sort_by_highest_values else 'lowest'} {col}:")
for n, s in zip(names, svals): print(f"  {s:<{w2}} ----> {n:<{w1}}")
print("\npymol", *sel['rfd3_cifgz_path'])

### Filter

In [ ]:
#############################################
### LOAD COMBINED JSON & FILTER METRICS   ###
#############################################

### COMBINED JSON PATH FROM ABOVE ###
output_combined_json_path = f"{rfdiffusion3_out_dir}i1/combined_stats.json"

# The combined statistics file is written after RFdiffusion3 inference and the
# JSON-merge step above. Run those first; this cell reads their output.
if not os.path.exists(output_combined_json_path) or not json.load(open(output_combined_json_path)):
    raise FileNotFoundError(
        f"No RFdiffusion3 statistics found at {output_combined_json_path}.\n"
        f"  Run the inference commands emitted above, then the JSON-merge cell,\n"
        f"  before running this filtering step."
    )

### HELPER TO BUILD EXTREME FUNCTIONS ###
def take_json_nest_min_or_max(prefix: str, agg: str = 'max'):
    def fn(df: pd.DataFrame) -> pd.Series:
        cols = df.filter(regex=rf'^{prefix}\.').columns
        if agg == 'max':
            return df[cols].max(axis=1)
        elif agg == 'min':
            return df[cols].min(axis=1)
        else:
            raise ValueError("agg must be 'max' or 'min'")
    fn.__name__ = f"{agg}_{prefix.split('.')[-1]}"
    return fn

### FILTERS ###
conditions = [
    ### sidechain quality ###
    (take_json_nest_min_or_max('metrics.join_point_rmsd_by_token', 'max'), '<', 0.8),
    #('metrics.insertion.mae', '<', 0.6), # MAE = (no alignment) | really just for debugging
    ('metrics.insertion.rmcd', '<', 0.6), # RMCD = (centers but no rotation) - should agree w RMSD
    ('metrics.insertion_rmsd', '<', 0.6), # RMSD = (full optimal alignment) - best metric
    ('metrics.join_point_rmsd', '<', 0.6),
    ('metrics.n_conjoined_residues', '<=', 0), # | NOT SURE WHAT THIS MEANS
    ### diversity content ###
    ('metrics.alanine_content', '<', 0.4),
    ('metrics.glycine_content', '<', 0.3),
    #('metrics.num_ss_elements', '<', 9),
    ('metrics.non_loop_fraction', '>', 0.4),
    ('metrics.loop_fraction', '<', 0.6),
    ('metrics.helix_fraction', '>', 0.05),
    ('metrics.sheet_fraction', '>', 0.000),

    ### backbone quality ###
    ('metrics.max_ca_deviation', '<', 4.0), # 3.8 Å between consecutive Cα atoms is ideal
    ('metrics.n_chainbreaks', '<=', 1),
    ('metrics.n_clashing.interresidue_clashes_w_sidechain', '<=', 3), # includes any sc-sc clash, usually resolved by seq design
    ('metrics.n_clashing.interresidue_clashes_w_backbone', '<=', 0),
   # ('metrics.n_clashing.ligand_min_distance', '>', 2), # backbone nearest dist with ligand
   # ('metrics.radius_of_gyration', '<', 18), # NON-NORMALIZED by number of residues (N)
    ('metrics.radius_of_gyration.norm_by_globularity', '<', 2.75), # NORMALIZED (N^(1/3))
    #('metrics.radius_of_gyration.norm_by_coil', '<', 18), # NORMALIZED (N^(0.58))
    ('metrics.radius_of_gyration.norm_by_ideal_sphere', '<', 0.86), # NORMALIZED by ideal protein sphere | Values ~~1 = very compact / sphere-like &&& Values ≫1 = elongated / dumbbell / multi-lobed
]

### SORT ###
number_of_sorted_to_print = 10
metric_to_sort            = "metrics.join_point_rmsd" # OR FUNCTION --> take_json_nest_min_or_max('metrics.join_point_rmsd_by_token','max')    # e.g. 'metrics.insertion_rmsd', 'metrics.join_point_rmsd', etc.
sort_by_highest_values    = True                         # True → top N highest values; False → bottom N lowest values

### FILTERING LOGIC ###
with open(output_combined_json_path, 'r') as f:
    records = json.load(f)
df    = pd.json_normalize(records, sep='.')
total = len(df)
print(f"Total records: {total}\n")

# prepare aligned printing
labels      = [(col.__name__ if callable(col) else col) for col, *_ in conditions]
label_width = max(len(str(l)) for l in labels)
opval_strs  = [f"{op} {val}" for *_, op, val in conditions]
opval_width = max(len(s) for s in opval_strs)
num_width   = len(str(total))
op_funcs = {'<': operator.lt, '<=': operator.le, '>': operator.gt, '>=': operator.ge}

# Print each filter’s pass count, aligned
for (col, op, val), label, opval in zip(conditions, labels, opval_strs):
    series = col(df) if callable(col) else df[col]
    mask   = op_funcs[op](series, val)
    count  = mask.sum()
    pct    = count / total * 100
    print(f"[{label:<{label_width}} {opval:<{opval_width}} ]:  "
          f"{count:>{num_width}} / {total:<{num_width}}  ({pct:6.3f}%)")

# Combined filter
combined_mask = pd.Series(True, index=df.index)
for col, op, val in conditions:
    series        = col(df) if callable(col) else df[col]
    combined_mask &= op_funcs[op](series, val)

combined_count = combined_mask.sum()
combined_pct   = combined_count / total * 100
print(f"\nPassed All Conditions → {combined_count} / {total} ({combined_pct:.3f}%)")

# keep the filtered DataFrame
rfd3_json_df_filtered = df[combined_mask]

### SORT PASSING STRUCTURES BY A VALUE & PRINT THEIR PATHS ###
df_tmp = (rfd3_json_df_filtered.assign(_sv=lambda d: metric_to_sort(d)).sort_values('_sv', ascending=not sort_by_highest_values)) if callable(metric_to_sort) else rfd3_json_df_filtered.sort_values(metric_to_sort, ascending=not sort_by_highest_values)
col = '_sv' if callable(metric_to_sort) else metric_to_sort

sel = df_tmp.head(number_of_sorted_to_print)
names = [os.path.basename(p) for p in sel['rfd3_cifgz_path']]
svals = [f"{v:.4f}" for v in sel[col]]
w1, w2 = max(map(len, names)), max(map(len, svals))

print(f"\n{'TOP' if sort_by_highest_values else 'BOTTOM'} {number_of_sorted_to_print} "
      f"{'highest' if sort_by_highest_values else 'lowest'} {col}:")
for n, s in zip(names, svals): print(f"  {s:<{w2}} ----> {n:<{w1}}")
print("\npymol", *sel['rfd3_cifgz_path'])

### Copy Filtered Files 

In [ ]:
#########################################
### COPY FILTERED JSON & CIF.GZ FILES ###
#########################################

### WHERE TO COPY FILES ###
copy_dest_dir = f"{rfdiffusion3_out_dir}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/"
os.makedirs(copy_dest_dir, exist_ok=True)

### COPYING LOGIC ###
n_rows = len(rfd3_json_df_filtered)
needed = n_rows * 2
print(f"Need to copy {needed} files ({n_rows} JSON + {n_rows} CIFs)")

copied = 0
for _, row in rfd3_json_df_filtered.iterrows():
    subdir = row['subdirectory']
    for key, is_cif in [('rfd3_json_path', False), ('rfd3_cifgz_path', True)]:
        src = row[key]
        base = os.path.basename(src)
        if is_cif and base.lower().endswith('.cif.gz'):
            name = base[:-7]
            ext  = '.cif.gz'
        else:
            name, ext = os.path.splitext(base)
        dest_name = f"{name}_{subdir}{ext}"
        dest_path = os.path.join(copy_dest_dir, dest_name)
        shutil.copy(src, dest_path)
        copied += 1

print(f"Copied {copied}/{needed} files to {copy_dest_dir}")

### Generate Rosetta `.params` Files for the Ligand

The scaffold-analysis step scores each design with PyRosetta, which needs a
Rosetta `.params` file for the ligand: neither `TSA` nor `SZD` is among the
~1,500 residue types Rosetta ships, so a pose containing one cannot be loaded
without `-extra_res_fa`. `process_diffusion3_outputs.py` takes it via `--params`.

The cell below builds the command that generates the parameter file from the
theozyme you already prepared, using
[`Scripts/theozyme_and_ligand_handling/ligands_to_params__UNIFIED.py`](../Scripts/theozyme_and_ligand_handling/ligands_to_params__UNIFIED.py).

> **Requires Rosetta.** This calls Rosetta's `molfile_to_params.py`, which is not
> redistributed here. Set `ROSETTA` to your Rosetta root (or `MOLFILE_TO_PARAMS`
> to the script itself) first:
>
> ```bash
> export ROSETTA=/path/to/rosetta/main
> python Scripts/repo_paths.py     # confirms what resolved
> ```
>
> Needed once per ligand; the resulting `.params` and `.pdb` are reusable.


In [ ]:
########################################################################
### GENERATE THE ROSETTA .params FILE FOR THIS TUTORIAL'S LIGAND     ###
########################################################################

### INPUTS ###
# Any theozyme PDB containing the ligand works; the pre-ORI reference is
# cleanest. The RFdiffusion3 worked example uses the 1QJI zinc protease (TSA).
params_input_pdb = f"{theozymes_dir}from_PDB_structure/step2__splitting/pdb_00001qji__theozyme_HEHH__lig_TSA.pdb"
params_ligand    = "TSA"

### OUTPUT ###
# params_files_dir is defined in the initialization cell and is what the
# scaffold-analysis cells below pass to --params.
params_output_dir = params_files_dir

### CONSTANTS ###
params_script = f"{scripts_dir}theozyme_and_ligand_handling/ligands_to_params__UNIFIED.py"

### BUILD THE COMMAND ###
command = (
    f"python {params_script} "
    f"--input_single_pdb {params_input_pdb} "
    f"--ligands_to_extract_via_3letter_code {params_ligand} "
    f"--desired_ligand_3letter_code {params_ligand} "
    f"--output_dir_for_params_stuff {params_output_dir} "
    f"--preserve_pdb_ligand_atom_order "
)

print("### COMMAND TO GENERATE THE LIGAND .params ###\n")
print(command)
print()
print(f"# Writes {params_output_dir}{params_ligand}.params (and {params_ligand}.pdb).")
print(f"# The scaffold-analysis cells below pass this via --params.")
print("#")
print("# Requires Rosetta's molfile_to_params.py -- check it resolves with:")
print(f"#     ROSETTA=/path/to/rosetta/main python {scripts_dir}repo_paths.py")


## III.C Additional Scaffold Filters

### Run Analysis

**NOTE:** Cannot have the reference pdb contain an ORI Token!

In [ ]:
#####################################################################
### RUN PROCESS RF-FLOW OUTPUTS (PARAMETERIZED PER-LIGAND CONFIG) ###
#####################################################################

### CONFIGURATIONS ###
#   - ref_catres and ligand_exposed_atoms are now OPTIONAL. - You can give them as a space-separated string ("A94 A96") or as a list (["A94", "A96"]).
run_configs = [
    {"ligand": "TSA", "pre_lig_frag1": "theozyme_HEHH__", "post_lig_frag1": "", "ref_catres": "", "ligand_exposed_atoms": ""},
]
# ligand_exposed_atoms names the ligand atoms that should stay solvent-facing
# (with --exposed_atom_SASA as the threshold). It is left empty here because the
# right choice depends on your chemistry -- for a substrate that must reach the
# active site from bulk, list the leaving-group atoms. TSA atom names are ZN1,
# P1, N1-N6, O1-O9 and C1-C36; see the input PDB.

### VARIABLES ###
scaffolds_to_analyze_DIR   = f"{rfdiffusion3_out_dir}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/"
combined_input_ligands_DIR = f"{theozymes_dir}from_PDB_structure/step2__splitting/"  # pre-ORI references
params_path_DIR            = params_files_dir

# Optional numeric thresholds: set to None to skip passing that flag
exposed_atom_SASA = 1.0     # None → do not pass --exposed_atom_SASA
cart_bonded       = 1.0     # None → do not pass --cart_bonded
fa_dun            = 1.0     # None → do not pass --fa_dun

# Optional boolean flags controlling script behavior
analyze_only                       = True   # True  → add --analyze  (no filtering/moving)
apply_loop_catres_filter           = False  # True  → use default behavior (filter) | False → pass --loop_catres to DISABLE filter
fix_unmatched_remark_lines_to_lig  = True   # True  → add --fix_unmatched_remark_lines_to_lig

# Other optional flags
partial_diffusion = False          # True  → add --partial
nproc             = None           # e.g. 8; None → let script auto-detect

### CONSTANTS ###
### EXECUTION ENVIRONMENT ###
# None -> run with the interpreter this notebook is using (conda activate zinc_hydro).
# To run inside a container instead, set a .sif path here or export ZINC_HYDRO_SIF.
# Build one with:  apptainer build zinc_hydro.sif Environment/zinc_hydro.def
apptainer = None
runner = " ".join(env_config.resolve_runner(apptainer))
script = f"{scripts_dir}process_diffusion3_outputs.py"   # vendored into this repository
file_extn = ".cif.gz"

### GENERATE & PRINT COMMANDS ###
for cfg in run_configs:
    ligand = cfg["ligand"]
    frag   = cfg["pre_lig_frag1"]
    post   = cfg.get("post_lig_frag1", "")

    label = f"{ligand}_{frag}" + (f"_{post}" if post else "")     # build label, avoid trailing underscore if post is empty
    post_pattern = f"*{post}" if post else ""     # build wildcard patterns, avoid double-asterisk if post is empty

    pdb_path    = f"{scaffolds_to_analyze_DIR}*{frag}*{ligand}{post_pattern}*{file_extn}"
    ref_path    = f"{combined_input_ligands_DIR}*{frag}*{ligand}{post_pattern}*.pdb"
    params_path = f"{params_path_DIR}{ligand}.params"

    cmd_parts = [runner, script,
        "--pdb", pdb_path,
        "--params", params_path,
    ]

    # Optional numeric thresholds
    if cart_bonded is not None:
        cmd_parts += ["--cart_bonded", str(cart_bonded)]
    if fa_dun is not None:
        cmd_parts += ["--fa_dun", str(fa_dun)]

    # Optional ligand-exposed atoms + SASA (per-config)
    lig_exp_atoms = cfg.get("ligand_exposed_atoms", None)
    if lig_exp_atoms:
        if isinstance(lig_exp_atoms, str):
            lig_exp_atoms_tokens = lig_exp_atoms.split()
        else:
            lig_exp_atoms_tokens = list(lig_exp_atoms)
        cmd_parts += ["--ligand_exposed_atoms", *lig_exp_atoms_tokens]

        if exposed_atom_SASA is not None:
            cmd_parts += ["--exposed_atom_SASA", str(exposed_atom_SASA)]

    # Optional ref_catres (per-config)
    ref_catres_cfg = cfg.get("ref_catres", None)
    if ref_catres_cfg:
        if isinstance(ref_catres_cfg, str):
            ref_catres_tokens = ref_catres_cfg.split()
        else:
            ref_catres_tokens = list(ref_catres_cfg)
        cmd_parts += ["--ref_catres", *ref_catres_tokens]

    if not apply_loop_catres_filter:     #   - apply_loop_catres_filter == True  → do nothing (keep default behavior) | apply_loop_catres_filter == False → pass --loop_catres to disable filter
        cmd_parts.append("--loop_catres")
    if analyze_only:
        cmd_parts.append("--analyze")
    if fix_unmatched_remark_lines_to_lig:     # Fix REMARK 666 unmatched targets
        cmd_parts.append("--fix_unmatched_remark_lines_to_lig")
    if partial_diffusion:     # Partial diffusion flag
        cmd_parts.append("--partial")
    if nproc is not None:     # nproc: let the script auto-detect if None
        cmd_parts += ["--nproc", str(nproc)]
    cmd = " ".join(cmd_parts) # Final command string

    print(cmd)
    print(f"mv diffusion_analysis.sc rfdiffusion3_analysis_{label}.sc", f"\n")


Remember, if you are seeing bugs - it is probably because your reference PDBs have ORI tokens in them. I also had a problem with the ligand naming, you need to stop that problem early in this stage.

In [ ]:
############################################################################
### COMBINE ALL VALID RFdiffusion3 .SC FILES (PYTHON EXECUTION; NO BASH) ###
############################################################################

### INPUTS ###
analysis_dir = f'{rfdiffusion3_out_dir}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/'  # change as needed

### FIND FILES ###
sc_pattern = os.path.join(analysis_dir, "rfdiffusion3_analysis_*.sc")
all_files = sorted(glob.glob(sc_pattern))

included_files = []
excluded_files = []

# Decide which files have any valid rows (fewer than 9 NaNs per row)
for file in all_files:
    try:
        df = pd.read_csv(file, sep=r'\s+', engine='python')
        df_clean = df[df.isna().sum(axis=1) < 9]
        if len(df_clean) > 0:
            included_files.append(file)
        else:
            excluded_files.append(file)
    except Exception:
        excluded_files.append(file)

# Report what we found
print(f"# Total .sc files found: {len(all_files)}")
print(f"# Excluding {len(excluded_files)} files with no valid rows:")
for f in excluded_files:
    print(f"  - {os.path.basename(f)}")
print(f"# Including {len(included_files)} files:")
for f in included_files:
    print(f"  - {os.path.basename(f)}")

if not included_files:
    print("# No files to combine.")
else:
    ### OUTPUT ###
    combined_file = os.path.join(analysis_dir, "combined_RFdiffusion3_analysis.sc")

    # ---------- helpers ----------
    def count_rows(path: str) -> int:
        # counts total lines (including header)
        with open(path, "r") as fh:
            return sum(1 for _ in fh)

    def num_fields(line: str) -> int:
        return len(line.strip().split())

    def print_dimensions(path: str):
        rows = count_rows(path)
        with open(path, "r") as fh:
            header = fh.readline()
        cols = num_fields(header)
        print(f"{path} - Rows: {rows}, Columns: {cols}")

    # ---------- combine ----------
    with open(included_files[0], "r") as f0:
        header = f0.readline()

    with open(combined_file, "w") as out:
        out.write(header)
        # append all data lines (skip each file header)
        for fp in included_files:
            with open(fp, "r") as fin:
                _ = fin.readline()  # skip header
                for line in fin:
                    out.write(line)

    print(f"\n[INFO] Wrote combined file:\n  {combined_file}\n")

    # ---------- print dimensions ----------
    for fp in included_files:
        print_dimensions(fp)
    print_dimensions(combined_file)

    # ---------- verify row counts ----------
    expected_rows = sum(count_rows(fp) for fp in included_files) - (len(included_files) - 1)  # remove duplicate headers
    actual_rows = count_rows(combined_file)
    if expected_rows == actual_rows:
        print("\n[CHECK] Row count matches expectation.")
    else:
        print(f"\n[CHECK] Mismatch in row count! Expected: {expected_rows}, Found: {actual_rows}")

    # ---------- verify column count consistency ----------
    expected_cols = num_fields(header)
    bad_lines = 0
    with open(combined_file, "r") as fh:
        for i, line in enumerate(fh, start=1):
            if not line.strip():
                continue
            nf = num_fields(line)
            if nf != expected_cols:
                bad_lines += 1
                if bad_lines <= 10:
                    print(f"[WARN] Column mismatch at line {i}: NF={nf} (expected {expected_cols})")
                elif bad_lines == 11:
                    print("[WARN] (More column mismatches exist; suppressing further warnings.)")

    if bad_lines == 0:
        print("[CHECK] Column count is consistent.")
    else:
        print(f"[CHECK] Column count mismatch! Total mismatching lines: {bad_lines}")


### Plot the Distributions & Look at Filtering Criteria/Pass Rates

These steps are technically optional but highly recommended - it is a manual way for you to get a sense of what values to use for when you run the actual filtering script without the `--analyze` flag. 

In [ ]:
##############################################
### PLOT THE DISTRIBUTION OF ALL THE STATS ###
##############################################

# INPUT COMBINED .SC FILE TO LOAD
file_path = f'{rfdiffusion3_out_dir}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/combined_RFdiffusion3_analysis.sc'
# Written by the scorefile-combining cell above, which merges the per-group
# rfdiffusion3_analysis_*.sc files produced by the scaffold-analysis commands.
if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"No combined analysis scorefile at {file_path}.\n"
        f"  Run the scaffold-analysis commands emitted above, then the cell that\n"
        f"  combines their .sc files, before running this one."
    )


# Load the data
df = pd.read_csv(file_path, sep=r'\s+', engine='python')

##################
### PLOT STUFF ###
##################

# Define numerical columns, excluding 'loop_at_motif' if it exists
numerical_cols = df.select_dtypes(include=[np.number]).columns
numerical_cols = numerical_cols[numerical_cols != 'loop_at_motif']

quartile_colors = ['skyblue', 'mediumseagreen', 'gold', 'salmon']

# -------------------------- #
#  ADJUSTABLE FONT SCALES    #
# -------------------------- #
title_fontsize = 20
label_fontsize = 18
tick_fontsize  = 16

# Set up the grid for the subplots
num_cols_per_row = 6
num_rows = int(np.ceil(len(numerical_cols) / num_cols_per_row))
fig, axes = plt.subplots(num_rows, num_cols_per_row, figsize=(6 * num_cols_per_row, 6 * num_rows))

# Flatten axes for easier iteration; remove excess axes if needed
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    ax = axes[idx]
    
    # 1) Draw the KDE line with a non-zero linewidth
    sns.kdeplot(
        data=df, x=col, ax=ax, fill=False, 
        linewidth=2, color='black'
    )
    
    # 2) Calculate quartiles
    q1, q2, q3 = df[col].quantile([0.25, 0.50, 0.75])
    
    # 3) Fill the area under the KDE curve in quartiles (if a line was created)
    if ax.lines:
        line = ax.lines[-1]
        x_data, y_data = line.get_data()
        
        ax.fill_between(
            x_data, y_data,
            where=(x_data < q1),
            color=quartile_colors[0], alpha=0.5
        )
        ax.fill_between(
            x_data, y_data,
            where=((x_data >= q1) & (x_data < q2)),
            color=quartile_colors[1], alpha=0.5
        )
        ax.fill_between(
            x_data, y_data,
            where=((x_data >= q2) & (x_data < q3)),
            color=quartile_colors[2], alpha=0.5
        )
        ax.fill_between(
            x_data, y_data,
            where=(x_data >= q3),
            color=quartile_colors[3], alpha=0.5
        )
    
    # Set title and labels with adjustable font sizes
    ax.set_title(f'Distribution of {col}', fontsize=title_fontsize, weight='semibold')
    ax.set_xlabel(f'{col}', fontsize=label_fontsize, weight='semibold')
    ax.set_ylabel('Density', fontsize=label_fontsize, weight='semibold')
    
    # Tick parameters: size, direction, etc.
    ax.tick_params(
        axis='both', which='major', 
        labelsize=tick_fontsize, 
        direction='in', length=5, width=1.5,
        top=True, right=True  # turn on top & right ticks
    )
    
    # Make spines visible on all four sides and set their width/color
    for side in ['left', 'right', 'top', 'bottom']:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(2)
        ax.spines[side].set_edgecolor('black')

# Hide any excess axes
for ax in axes[len(numerical_cols):]:
    ax.remove()

plt.tight_layout()

# Save the figure to the specified directory
save_path = f'{graphs_dir}RFdiffusion3_stats_analysis_PRODUCTION.png'
#plt.savefig(save_path, dpi=300)
plt.show()

The next cell is setup quite ideally so that you can look at the extrema for certain filtering criteria. Just choose a column to sort by and if you want it ascending or descending. Then it will print out the n_example of pdb files that passed all the filters and are the extrema for the sorting column you picked - you can use this to adjust cutoffs and inform the values you want to use. Just copy the list of pdb files and then do `pymol {paste copied list of pdb files}` in your terminal and you will see the pdb files ranked in order of the filtering column.

**NOTE:** I will probably go fairly light with this filtering, especially SASA, and really focus in after predesign.

In [ ]:
#########################################
### INVESTIGATE IDEAL FILTER CRITERIA ###
#########################################

### SHOW THE FIRST FEW ROWS OF THE FILTERED DF? ###
show_df_head = True

### SORTING PARAMETERS FOR FILTERED DF ###
sorting_col = "cart_bonded_avg"
sorting_ascending = False # smallest first = true
n_example = 20  # How many top examples to display

### INPUT DATAFRAME .SC FROM ANALYSIS FILTERING STEP ###
file_path = f'{rfdiffusion3_out_dir}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/combined_RFdiffusion3_analysis.sc'
# Written by the scorefile-combining cell above, which merges the per-group
# rfdiffusion3_analysis_*.sc files produced by the scaffold-analysis commands.
if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"No combined analysis scorefile at {file_path}.\n"
        f"  Run the scaffold-analysis commands emitted above, then the cell that\n"
        f"  combines their .sc files, before running this one."
    )


### LOGIC ###
df = pd.read_csv(file_path, sep=r'\s+', engine='python') # Meant for .sc files
initial_count = len(df)
df_clean = df[df.isna().sum(axis=1) < 9].copy()
removed_count = initial_count - len(df_clean)
print(f"Removed {removed_count} rows with 9 or more NaN values.")
print()
df = df_clean

### (OPTIONAL) CONSIDER ONLY ROWS WITH A CERTAIN SUBSTRING ###
description_filter = ""  # e.g. "frag1"
if description_filter:
    before_desc = len(df)
    df = df[df['description'].str.contains(description_filter, na=False)]
    desc_count = len(df)
    print(f"Filtered by description ‘{description_filter}’: removed {before_desc - desc_count} rows.")

### MANDATORY (HARD-CODED) FILTERS ###
chainbreak = 4.5     # CA-distance threshold
lig_dist   = 2.0     # Clash distance with ligand
rCA_nonadj = 3.0     # Non-adjacent residues cannot be closer than this
bondlen_dev = 0.1  # Allowed bond-length deviation

### OPTIONAL FILTERS ###
loop_frac       = 0.50   # Usually 0.3
longest_helix   = 46     # Max helix length
rog             = 17.5   # Radius of gyration
term_mindist    = 5      # Terminus distance from ligand?
cart_bonded_avg = 40     # sidechain quality | smaller = better | recommended ~5
fa_dun_avg      = 10     # sidechain quality | smaller = better | recommended ~15

### SASA UPPER + LOWER FILTERS ###
SASA_rel_lower = 0.01   # Scale 0-1
SASA_rel_upper = 0.19    # Scale 0-1

### SASA EXPOSED ATOM FILTERS ###
SASA_exposed_atoms_lower = 30
SASA_exposed_atoms_upper = 66

### DEFINE FILTERS (adjust or comment out as desired) ###
filters = {
    ("chainbreak", "<="): chainbreak,
    ("lig_dist", ">="): lig_dist,
    ("rCA_nonadj", ">="): rCA_nonadj,
    ("bondlen_dev", "<="): bondlen_dev,
    ("loop_frac", "<="): loop_frac,
    ("longest_helix", "<="): longest_helix,
    #("rog", "<="): rog,
    ("SASA_rel", ">="): SASA_rel_lower,
    ("SASA_rel", "<="): SASA_rel_upper,
    ("SASA_exposed_atoms", ">="): SASA_exposed_atoms_lower,
    #("SASA_exposed_atoms", "<="): SASA_exposed_atoms_upper,
    ("term_mindist", ">="): term_mindist,
    
    ("cart_bonded_avg", "<="): cart_bonded_avg,
    ("fa_dun_avg", "<="): fa_dun_avg,
}

def filtering_df(dataframe_name, filters, target_df, print_stat=True):
    """
    Apply a set of filters to a DataFrame and print how many rows pass each one.
    Returns the filtered DataFrame.
    """
    if print_stat:
        print(f"[{dataframe_name}] ({len(target_df)} designs)")
    
    converted_filters = []
    for (key, operator_str), cutoff_value in filters.items():
        # Build a condition string, e.g. "target_df['chainbreak'] <= 4.5"
        filter_str = f"target_df['{key}'] {operator_str} {cutoff_value}"
        converted_filters.append(filter_str)
        
        if print_stat:
            pass_mask = eval(filter_str)
            count_passed = pass_mask.sum()
            pct_passed = (count_passed / len(target_df)) * 100
            print(
                f"# [ {key.ljust(19)} {operator_str.ljust(2)} "
                f"{str(cutoff_value).ljust(6)}]: "
                f"{str(count_passed).ljust(6)} ({pct_passed:.1f}%)"
            )
    
    # Combine all filters with logical AND
    final_mask = eval(" & ".join([f"({condition})" for condition in converted_filters]))
    filtered_df = target_df[final_mask]
    
    if print_stat:
        total_passed = len(filtered_df)
        pct_passed = (total_passed / len(target_df)) * 100
        print(f"# [             ALL              ]: {str(total_passed).ljust(6)} ({pct_passed:.1f}%)")
    
    return filtered_df

### APPLY FILTERS ###
filtered_RFdiffusion3_df = filtering_df("RFdiffusion3", filters, df)
filter_rate = (len(filtered_RFdiffusion3_df) / len(df)) * 100.0

print(f"\n{len(filtered_RFdiffusion3_df)} of {len(df)} designs ({round(filter_rate, 3)}%) passed\n")

### OPTIONALLY SHOW FIRST n ROWS SORTED BY A SPECIFIC COLUMN ###
if show_df_head:
    print(f"# SORTED BY [{sorting_col}] IN [{'Descending' if not sorting_ascending else 'Ascending'}] ORDER:")
    filtered_RFdiffusion3_df.sort_values(by=sorting_col, ascending=sorting_ascending, inplace=True)
    top_descriptions = filtered_RFdiffusion3_df['description'].head(n_example).tolist()
    print("pymol " + ' '.join(top_descriptions))
    print('')
    #print(filtered_RFdiffusion3_df.head(n_example))
    print(filtered_RFdiffusion3_df[[sorting_col, "description"]].head(n_example))    # Only show the sorting column in the head

### Execute the Filtering

In [ ]:
#####################################################################
### RUN PROCESS RF-FLOW OUTPUTS (PARAMETERIZED PER-LIGAND CONFIG) ###
#####################################################################

### CONFIGURATIONS ###
#   - ref_catres and ligand_exposed_atoms are now OPTIONAL. - You can give them as a space-separated string ("A94 A96") or as a list (["A94", "A96"]).
run_configs = [
    {"ligand": "TSA", "pre_lig_frag1": "theozyme_HEHH__", "post_lig_frag1": "", "ref_catres": "", "ligand_exposed_atoms": ""},
]
# ligand_exposed_atoms names the ligand atoms that should stay solvent-facing
# (with --exposed_atom_SASA as the threshold). It is left empty here because the
# right choice depends on your chemistry -- for a substrate that must reach the
# active site from bulk, list the leaving-group atoms. TSA atom names are ZN1,
# P1, N1-N6, O1-O9 and C1-C36; see the input PDB.

### VARIABLES ###
scaffolds_to_analyze_DIR   = f"{rfdiffusion3_out_dir}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/"
combined_input_ligands_DIR = f"{theozymes_dir}from_PDB_structure/step2__splitting/"  # pre-ORI references
params_path_DIR            = params_files_dir

### PROTEIN QUALITY FILTERS ###
rog           = None
longest_helix = 46
loop_limit    = 0.5
lig_dist      = 2.0

### LIGAND-PROTEIN QUALITY FILTERS ###
SASA_limit           = 0.19  # upper limit
terminus_dist_limit  = 5.0   # lower limit

### SIDE CHAIN QUALITY FILTERS ###
bondlen_dev = 0.1

# Optional numeric thresholds: set to None to skip passing that flag
exposed_atom_SASA = 30      # None → do not pass --exposed_atom_SASA
cart_bonded       = 40      # None → do not pass --cart_bonded
fa_dun            = 10      # None → do not pass --fa_dun

# Optional boolean flags controlling script behavior
analyze_only                       = False  # True  → add --analyze  (no filtering/moving)
apply_loop_catres_filter           = False  # True  → use default behavior (filter) | False → pass --loop_catres to DISABLE filter
fix_unmatched_remark_lines_to_lig  = True   # True  → add --fix_unmatched_remark_lines_to_lig

# Other optional flags
partial_diffusion = False          # True  → add --partial
nproc             = None           # e.g. 8; None → let script auto-detect

### CONSTANTS ###
### EXECUTION ENVIRONMENT ###
# None -> run with the interpreter this notebook is using (conda activate zinc_hydro).
# To run inside a container instead, set a .sif path here or export ZINC_HYDRO_SIF.
# Build one with:  apptainer build zinc_hydro.sif Environment/zinc_hydro.def
apptainer = None
runner = " ".join(env_config.resolve_runner(apptainer))
script = f"{scripts_dir}process_diffusion3_outputs.py"   # vendored into this repository
file_extn = ".cif.gz"

### GENERATE & PRINT COMMANDS ###
for cfg in run_configs:
    ligand = cfg["ligand"]
    frag   = cfg["pre_lig_frag1"]
    post   = cfg.get("post_lig_frag1", "")

    label = f"{ligand}_{frag}" + (f"_{post}" if post else "")     # build label, avoid trailing underscore if post is empty
    post_pattern = f"*{post}" if post else ""     # build wildcard patterns, avoid double-asterisk if post is empty

    pdb_path    = f"{scaffolds_to_analyze_DIR}*{frag}*{ligand}{post_pattern}*{file_extn}"
    ref_path    = f"{combined_input_ligands_DIR}*{frag}*{ligand}{post_pattern}*.pdb"
    params_path = f"{params_path_DIR}{ligand}.params"

    cmd_parts = [runner, script, "--pdb", pdb_path, "--params", params_path,]

    # Protein quality filters
    if rog is not None:
        cmd_parts += ["--rog", str(rog)]
    if longest_helix is not None:
        cmd_parts += ["--longest_helix", str(longest_helix)]
    if loop_limit is not None:
        cmd_parts += ["--loop_limit", str(loop_limit)]
    if lig_dist is not None:
        cmd_parts += ["--lig_dist", str(lig_dist)]

    # Ligand-protein quality filters
    if SASA_limit is not None:
        cmd_parts += ["--SASA_limit", str(SASA_limit)]
    if terminus_dist_limit is not None:
        cmd_parts += ["--term_limit", str(terminus_dist_limit)]

    # Side chain quality filters
    if bondlen_dev is not None:
        cmd_parts += ["--bondlen_dev", str(bondlen_dev)]

    # Optional numeric thresholds
    if cart_bonded is not None:
        cmd_parts += ["--cart_bonded", str(cart_bonded)]
    if fa_dun is not None:
        cmd_parts += ["--fa_dun", str(fa_dun)]

    # Optional ligand-exposed atoms + SASA (per-config)
    lig_exp_atoms = cfg.get("ligand_exposed_atoms", None)
    if lig_exp_atoms:
        if isinstance(lig_exp_atoms, str):
            lig_exp_atoms_tokens = lig_exp_atoms.split()
        else:
            lig_exp_atoms_tokens = list(lig_exp_atoms)
        cmd_parts += ["--ligand_exposed_atoms", *lig_exp_atoms_tokens]

        if exposed_atom_SASA is not None:
            cmd_parts += ["--exposed_atom_SASA", str(exposed_atom_SASA)]

    # Optional ref_catres (per-config)
    ref_catres_cfg = cfg.get("ref_catres", None)
    if ref_catres_cfg:
        if isinstance(ref_catres_cfg, str):
            ref_catres_tokens = ref_catres_cfg.split()
        else:
            ref_catres_tokens = list(ref_catres_cfg)
        cmd_parts += ["--ref_catres", *ref_catres_tokens]

    if not apply_loop_catres_filter:     #   - apply_loop_catres_filter == True  → do nothing (keep default behavior) | apply_loop_catres_filter == False → pass --loop_catres to disable filter
        cmd_parts.append("--loop_catres")
    if analyze_only:
        cmd_parts.append("--analyze")
    if fix_unmatched_remark_lines_to_lig:     # Fix REMARK 666 unmatched targets
        cmd_parts.append("--fix_unmatched_remark_lines_to_lig")
    if partial_diffusion:     # Partial diffusion flag
        cmd_parts.append("--partial")
    if nproc is not None:     # nproc: let the script auto-detect if None
        cmd_parts += ["--nproc", str(nproc)]
    cmd = " ".join(cmd_parts) # Final command string
    print(cmd)

### OPTIONAL: Filter by Catalytic Residue Distance

Now I decided to do some filtering in the sequence space. Turns out, there are pretty distinct motifs for zinc binding: https://pmc.ncbi.nlm.nih.gov/articles/PMC3268031/ \
There is also a super amazing table here: https://febs.onlinelibrary.wiley.com/doi/epdf/10.1016/0014-5793%2894%2901079-X \
**HExxH** motif forming an α-helix \
**HExxHxxGFxHExxRxDR** furthermore... \
**HxxE(D)-aan-H** in carboxypeptidase family \
**HxD-aa12-H-aa12-H** matrix metalloprotease \
**HELLGH** in dipeptidyl peptidase & three kinds of monooxygenases

In [ ]:
###############################################################
### FILTER PDB FILES BY CATALYTIC RESIDUE SEQUENCE DISTANCE ###
###############################################################

### INPUT VARIABLES ###
input_dir = f"{rfdiffusion3_out_dir}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/filtered_structures"
catalytic_residue_pair_types = ["HIS GLU"]#["HIS HIS", "HIS GLU"]  # Residue pairs to check (comma list)
max_sequence_distances = [1]  # Max sequence distance thresholds for each pair (comma list)
min_sequence_distances = [1]  # Min sequence distance thresholds for each pair (comma list)

### CONSTANTS ###
special_scripts_dir = f'{github_repo_dir}/Scripts/'  # scaffold_handling/ scripts live here

### INITIALIZE COMMAND ###
command = (f"python {script_path} "
           f"--input_dir_of_pdbs_with_remark666_lines {input_dir} "
           f"--catalytic_residue_pair_types " + " ".join(f'"{pair}"' for pair in catalytic_residue_pair_types) + " ")

# Add optional distance thresholds
if max_sequence_distances:
    command += "--max_sequence_distances " + " ".join(map(str, max_sequence_distances)) + " "
if min_sequence_distances:
    command += "--min_sequence_distances " + " ".join(map(str, min_sequence_distances)) + " "

### PRINT COMMAND ###
print("#" * 49)
print("### GENERATED COMMAND FOR FILTERING PDB FILES ###")
print("#" * 49)
print("")
print(command)
print()

# **IV. Predesign Scaffolds (Skipped Metric Monster)** 

## IV.A Execute Predesign (Cartesian Relax for Geometry Idealization)

### Run Predesign

In [ ]:
#################################################################
### STAGE 1: GEOMETRY IDEALIZATION (RFdiffusion3 → Idealized) ###
#################################################################

### INPUTS ###
input_pdb_structures_dir = f"{rfdiffusion3_out_dir}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/filtered_structures"  # input filtered structures
# Optional: if JSONs are in a different directory than PDBs, specify here
corresponding_json_dir = f"{rfdiffusion3_out_dir}i1/ZZZ_MERGED_PRELIM_FILTER_DIR_ZZZ/"  # e.g., f"{rfdiffusion3_out_dir}i1/" or None if JSONs are with PDBs
# Optional: override per-structure JSON name (rare). Default is auto-detect from PDB basename.
json_file = None  # e.g., "/path/to/specific.json" or None

### GROUP → LIGAND MAP & FRAGS ###
# NOTE: dict keys must be unique; if you want multiple ligands per group, use a list.
groups_to_ligands = {"theozyme_HEHH": ["TSA"]}
# frags are filename fragments that sit between the group token and the ligand
# code, i.e. the pattern is *<group>*<frag>*<ligand>*.pdb
frags = ["lig"]

### OUTPUTS ###
specific_idealization_output_dir = f"{predesign_out_dir}i1_stage1/"  # where idealized structures go
# Optional: override per-structure output path (rare). Default is <input_basename>_idealized.pdb in output_dir.
output_pdb = None  # e.g., "/path/to/custom_name.pdb" or None

### PARAMS FILE LOCATION ###
params_files_dir = params_files_dir

### CONSTANTS ###
### EXECUTION ENVIRONMENT ###
# None -> run with the interpreter this notebook is using (conda activate zinc_hydro).
# To run inside a container instead, set a .sif path here or export ZINC_HYDRO_SIF.
# Build one with:  apptainer build zinc_hydro.sif Environment/zinc_hydro.def
apptainer_path = None
runner = " ".join(env_config.resolve_runner(apptainer_path))
special_scripts_dir = f'{github_repo_dir}/Scripts/'  # scaffold_handling/ scripts live here
script_path         = f'{special_scripts_dir}scaffold_handling/idealize_rfdiffusion3_geometry__MAIN.py'

### GEOMETRY IDEALIZATION PARAMETERS (optional - can adjust if needed) ###
mobile_radius      = 12.5    # Optional; default: 10.0 Å
coord_cst_weight   = 750.0   # Optional; default: 100.0 (tight constraints)
coord_cst_stdev    = 0.02    # Optional; default: 0.01 Å (very tight tolerance)
cart_bonded_weight = 0.7     # Optional; default: 0.5
fastrelax_cycles   = 2       # Optional; default: 2
min_tolerance      = 0.0001  # Optional; default: 0.0001

### PROTOCOL FLAGS (optional) ###
idealize_ss    = False  # Optional flag; default: False
skip_fastrelax = False  # Optional flag; default: False (set True to only minimize)
skip_minimize  = False  # Optional flag; default: False

### QUICK LOGIC ###
all_ligands = sorted({lig for ligs in groups_to_ligands.values() for lig in ligs})
def _find_params(lig):
    """Locate a ligand's .params, with an error that says how to make one."""
    matches = sorted(glob.glob(os.path.join(params_files_dir, f"*{lig}*.params")))
    if not matches:
        raise FileNotFoundError(
            f"No .params file for ligand '{lig}' in {params_files_dir}\n"
            f"  Run the ligand .params generation cell earlier in this notebook,\n"
            f"  then run its printed command in a terminal."
        )
    return matches[0]

params_files = {lig: _find_params(lig) for lig in all_ligands}
os.makedirs(specific_idealization_output_dir, exist_ok=True)

### GENERATE COMMANDS ###
commands_name = "predesign_i1_stage1"
commands = []
for g, lig_list in groups_to_ligands.items():
    for lig in lig_list:
        for f in frags:
            pattern = f"*{g}*{f}*{lig}*.pdb"
            for pdb_file in sorted(glob.glob(os.path.join(input_pdb_structures_dir, pattern))):
                params_file = params_files[lig]
                # Required: --pdb # Optional: --params, --json, --corresponding_json_dir, --output, --output_dir, --mobile_radius, --coord_cst_weight, --coord_cst_stdev, --cart_bonded_weight,  --idealize_ss, --skip_fastrelax, --skip_minimize    
                cmd = f"{runner} {script_path} --pdb {pdb_file} "
                # Practically required (recommended): params
                if params_file is not None:
                    cmd += f"--params {params_file} "
                # Auto-detected JSON (only pass if overriding)
                if json_file is not None:
                    cmd += f"--json {json_file} "
                if corresponding_json_dir is not None:
                    cmd += f"--corresponding_json_dir {corresponding_json_dir} "
                # Output controls (only pass if overriding; output_dir is recommended)
                if output_pdb is not None:
                    cmd += f"--output {output_pdb} "
                if specific_idealization_output_dir is not None:
                    cmd += f"--output_dir {specific_idealization_output_dir} "
                # Geometry parameters (only pass if overriding defaults)
                if mobile_radius != 10.0:
                    cmd += f"--mobile_radius {mobile_radius} "
                if coord_cst_weight != 100.0:
                    cmd += f"--coord_cst_weight {coord_cst_weight} "
                if coord_cst_stdev != 0.01:
                    cmd += f"--coord_cst_stdev {coord_cst_stdev} "
                if cart_bonded_weight != 0.5:
                    cmd += f"--cart_bonded_weight {cart_bonded_weight} "
                if fastrelax_cycles != 2:
                    cmd += f"--fastrelax_cycles {fastrelax_cycles} "
                if min_tolerance != 0.0001:
                    cmd += f"--min_tolerance {min_tolerance} "
                # Protocol flags (only pass when True)
                if idealize_ss:
                    cmd += "--idealize_ss "
                if skip_fastrelax:
                    cmd += "--skip_fastrelax "
                if skip_minimize:
                    cmd += "--skip_minimize "
                commands.append(cmd.strip())
commands.sort()

### OUTPUT & WRITE COMMANDS FILE ###
commands_file_path = os.path.join(cmds_dir, f"{commands_name}")
with open(commands_file_path, "w") as f:
    f.write("\n".join(commands) + "\n")
print("### COMMANDS FILE ###")
print(commands_file_path)
print("\nNumber of Commands =", len(commands))

### SETUP BATCH JOBS ###
qtime, cores, memory, queue = "03:30:00", "1", "8g", "cpu"
cmds_per_job = 5
job_name = os.path.basename(commands_file_path)
submit_file = f"{slurm_submit_dir}{job_name}.sh"
num_jobs = int(len(commands) / cmds_per_job)

print("Number of Jobs =", num_jobs)
print("Job Name =", job_name)
print("\nNavigate here:")
print("cd", specific_idealization_output_dir, "\n")

# make submit script
functions.submit_array_job(commands_file_path, qtime, cores, job_name, memory, submit_file, logs_dir, num_jobs + 1, cmds_per_job, queue)

### Parse JSONs

In [ ]:
############################################
### COMBINE JSON FILES INTO SINGLE JSON  ###
############################################

### INPUT DIRECTORY TO PARSE ###
rfd3_output_dir_to_parse = f"{predesign_out_dir}i1_stage1/"

### OUTPUT JSON PATH ###
output_combined_json_path = f"{predesign_out_dir}i1_stage1/combined_stats.json"

### HELPERS ###
def _safe_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

def _extract_catalytic_bond_geom_metrics(rec: dict):
    """
    From rec["catalytic_residues"]["details"][i]["bond_geometry"]:
      - max over all 'max_bond_deviation'
      - mean over all 'mean_bond_deviation'
    Returns (max_max_bond_deviation, mean_mean_bond_deviation) or (None, None).
    """
    cat = rec.get("catalytic_residues", {})
    details = cat.get("details", [])
    if not isinstance(details, list) or len(details) == 0:
        return None, None

    max_vals = []
    mean_vals = []
    for d in details:
        if not isinstance(d, dict):
            continue
        bg = d.get("bond_geometry", {})
        if not isinstance(bg, dict):
            continue
        mx = _safe_float(bg.get("max_bond_deviation", None))
        mn = _safe_float(bg.get("mean_bond_deviation", None))
        if mx is not None:
            max_vals.append(mx)
        if mn is not None:
            mean_vals.append(mn)

    max_max = max(max_vals) if max_vals else None
    mean_mean = (sum(mean_vals) / len(mean_vals)) if mean_vals else None
    return max_max, mean_mean


### PARSING & SORTING LOGIC ###
json_files = glob.glob(f"{rfd3_output_dir_to_parse}**/*.json", recursive=True)
json_files = sorted(json_files, key=lambda p: os.path.abspath(p))
json_files = [p for p in json_files if os.path.abspath(p) != os.path.abspath(output_combined_json_path)]

print(f"Found {len(json_files)} JSON files to parse")

all_records = []
skipped = 0

for path in json_files:
    try:
        with open(path, "r") as f:
            rec = json.load(f)
    except Exception as e:
        print(f"⚠️  Skipping {path!r}: failed to read JSON ({e})")
        skipped += 1
        continue

    if not isinstance(rec, dict):
        print(f"⚠️  Skipping {path!r}: top-level JSON is not an object")
        skipped += 1
        continue

    # extract catalytic bond-geometry metrics BEFORE dropping catalytic_residues
    max_max_bd, mean_mean_bd = _extract_catalytic_bond_geom_metrics(rec)

    # ensure global_metrics exists, then add our new metrics there
    global_metrics = rec.get("global_metrics", {})
    if not isinstance(global_metrics, dict):
        global_metrics = {}
    global_metrics["catalytic_max_max_bond_deviation"] = max_max_bd
    global_metrics["catalytic_mean_mean_bond_deviation"] = mean_mean_bd
    rec["global_metrics"] = global_metrics

    # drop catalytic_residues to save space
    rec.pop("catalytic_residues", None)

    # ✅ actually keep this record
    all_records.append(rec)

# write combined JSON
with open(output_combined_json_path, "w") as out:
    json.dump(all_records, out, indent=2)

print(f"Saved {len(all_records)} records → {output_combined_json_path} (skipped {skipped})")

### Filter

In [ ]:
###############################################
### LOAD COMBINED JSON & FILTER (PREDESIGN) ###
###############################################

### COMBINED JSON PATH FROM ABOVE ###
output_combined_json_path = f"{predesign_out_dir}i1_stage1/combined_stats.json"

### FILTERS (EDIT THESE THRESHOLDS FREELY) ###
conditions = [
    # core structural sanity
    ("global_metrics.num_chain_breaks", "<=", 0),
    ("global_metrics.ca_rmsd_overall", "<", 0.8),

    # clashes
    #("global_metrics.num_clashing_residues", "<", 8),
    ("global_metrics.num_catalytic_clashing", "<", 6),

    # catalytic geometry (new metrics you injected)
    ("global_metrics.catalytic_max_max_bond_deviation", "<=", 0.125),
    ("global_metrics.catalytic_mean_mean_bond_deviation", "<=", 0.025),

    ("global_metrics.mean_fixed_atom_displacement", "<=", 0.025),
    ("global_metrics.max_fixed_atom_displacement", "<", 0.05),

    # (optional) constraint/strain-ish
    ("global_metrics.mean_catalytic_cart_bonded", "<=", 7.5),
    # ("global_metrics.cart_bonded", "<", 380.0),

    # score (more negative is better, so usually "<")
    # ("global_metrics.total_score", "<", -150.0),
]

### SORT ###
number_of_sorted_to_print = 30
metric_to_sort            = "global_metrics.catalytic_max_max_bond_deviation"   # smaller (more negative) is better
sort_by_highest_values    = True                          # False -> best (lowest) for total_score

### LOAD ###
with open(output_combined_json_path, "r") as f:
    records = json.load(f)

df = pd.json_normalize(records, sep=".")
total = len(df)
print(f"Total records: {total}\n")

### FILTERING LOGIC ###
op_funcs = {"<": operator.lt, "<=": operator.le, ">": operator.gt, ">=": operator.ge}

# pretty printing widths
labels      = [col for col, *_ in conditions]
label_width = max(len(str(l)) for l in labels) if labels else 10
opval_strs  = [f"{op} {val}" for *_, op, val in conditions]
opval_width = max(len(s) for s in opval_strs) if opval_strs else 10
num_width   = len(str(total))

# print each filter’s pass count
for (col, op, val), opval in zip(conditions, opval_strs):
    series = df[col] if col in df.columns else pd.Series([None] * total, index=df.index)
    mask   = op_funcs[op](series, val)
    count  = int(mask.sum())
    pct    = (count / total * 100) if total else 0.0
    missing = int(series.isna().sum())
    print(f"[{col:<{label_width}} {opval:<{opval_width}} ]:  "
          f"{count:>{num_width}} / {total:<{num_width}}  ({pct:6.3f}%)   | missing={missing}")

# combined filter
combined_mask = pd.Series(True, index=df.index)
for col, op, val in conditions:
    series = df[col] if col in df.columns else pd.Series([None] * total, index=df.index)
    combined_mask &= op_funcs[op](series, val)

combined_count = int(combined_mask.sum())
combined_pct   = (combined_count / total * 100) if total else 0.0
print(f"\nPassed All Conditions → {combined_count} / {total} ({combined_pct:.3f}%)")

df_filt = df[combined_mask].copy()

### SORT + PRINT ###
if combined_count == 0:
    print("\nNo structures passed. Consider loosening thresholds or checking which columns are missing.")
else:
    df_tmp = df_filt.sort_values(metric_to_sort, ascending=not sort_by_highest_values)

    sel = df_tmp.head(number_of_sorted_to_print)

    # friendly name + path columns (these exist in your new JSON)
    name_col = "metadata.structure_name"
    pdb_col  = "metadata.pdb_path"

    names = sel[name_col].tolist() if name_col in sel.columns else ["(missing)"] * len(sel)
    paths = sel[pdb_col].tolist()  if pdb_col  in sel.columns else ["(missing)"] * len(sel)

    col = metric_to_sort
    svals = []
    for v in sel[col].tolist():
        try:
            svals.append(f"{float(v):.4f}")
        except Exception:
            svals.append(str(v))

    w1 = max(map(len, names)) if names else 10
    w2 = max(map(len, svals)) if svals else 10

    print(f"\n{'TOP' if sort_by_highest_values else 'BOTTOM'} {min(number_of_sorted_to_print, len(sel))} "
          f"{'highest' if sort_by_highest_values else 'lowest'} {col}:")
    for s, n, p in zip(svals, names, paths):
        print(f"  {s:<{w2}} ----> {n:<{w1}}   |   {p}")

    print("\n# quick copy/paste PyMOL command")
    print("pymol", *paths)

### Copy Filtered Files

In [ ]:
#########################################
### COPY FILTERED PDB STRUCTURES ###
#########################################

### WHERE TO COPY FILES ###
copy_dest_dir = f"{predesign_out_dir}i1_stage1/ZZZ_FILTERED_STRUCTURES_ZZZ/"
os.makedirs(copy_dest_dir, exist_ok=True)

### EXPECTED COLUMNS ###
pdb_col   = "metadata.pdb_path"
name_col  = "metadata.structure_name"

### COPYING LOGIC ###
n_rows = len(df_filt)
print(f"Need to copy {n_rows} PDB files")

copied = 0
missing = 0

for _, row in df_filt.iterrows():
    src = row.get(pdb_col, None)
    if not isinstance(src, str) or not os.path.exists(src):
        missing += 1
        continue

    base = os.path.basename(src)
    name = row.get(name_col, os.path.splitext(base)[0])

    # preserve uniqueness + readability
    dest_name = f"{name}.pdb"
    dest_path = os.path.join(copy_dest_dir, dest_name)

    shutil.copy(src, dest_path)
    copied += 1

print(f"Copied {copied}/{n_rows} PDBs to {copy_dest_dir}")
if missing:
    print(f"⚠️  Skipped {missing} entries due to missing PDB paths")